# FDS Challenge: My Notebook

## 0. Configuration

In [ ]:
import json
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm
from pathlib import Path

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split


DEBUG = False

pd.set_option("display.max_columns", None)  # keep it! it's usefull to make pandas print also the null columns
pd.set_option("display.width", None)


# --- PATHS ---
TRAIN_FILE_PATH        = "../dataset/train.jsonl"
TEST_FILE_PATH         = "../dataset/test.jsonl"
TYPE_CHART_PATH        = "../dataset/type_chart_gen2.csv"
STATS_MAP_PATH         = "../temp/stats_map.json"
SAMPLE_SUBMISSION_PATH = "../dataset/sample_submission.csv"
SUBMISSION_PATH        = "submission.csv"
REPORT_DIR             = "reports"  # One timestamped Markdown report per run.

Path("../temp").mkdir(exist_ok=True)


# --- FEATURES ---
TEAM_SIZE = 6

STATISTICS = ["hp", "atk", "def", "spa", "spd", "spe"]
HP_PCT_LOST_INTERVALS = ["0_10", "10_20", "20_30"]
MULTIPLIERS = [0, 0.25, 0.5, 1, 2, 4]
STATUSES = ["slp", "frz", "tox", "psn", "par", "brn"]
BOOSTS = ["atk", "def", "spa", "spd", "spe"]
# From player 1 POV
EFFECTS = {
    "good": [
        "beatup", "bide", "destinybond", "endure", "focusband",
        "focusenergy", "lockon", "magnitude", "mimic", "mindreader",
        "mysteryberry", "protect", "quickclaw", "safeguard", "substitute",
        "typechange"
    ],
    "bad": [
        "attract", "bind", "clamp", "confusion", "curse",
        "disable", "encore", "firespin", "foresight", "futuresight",
        "leechseed", "nightmare", "perish", "pursuit", "spite",
        "trapped", "whirlpool", "wrap"
    ]
}
STAB = 1.5

# --- HELD ITEMS DICTIONARIES & METRIC PRIORS ---
JUNK_ITEMS = {
    "berryjuice", "berry", "noitem", "pokeball", "firestone",
    "energypowder", "aspearberry", "gengarite", "alakazite", "blackbelt"
}

STATUS_BERRIES = {
    "miracleberry", "mintberry", "przcureberry", "psncureberry",
    "bitterberry", "lumberry", "chestoberry"
}

TYPE_BOOST_ITEMS = {
    "charcoal", "magnet", "blackbelt", "pinkbow", "polkadotbow",
    "mysticwater", "hardstone", "poisonbarb", "softsand", "silverpowder",
    "blackglasses", "sharpbeak", "spelltag", "miracleseed", "twistedspoon",
    "nevermeltice", "dragonscale", "dragonfang", "metalcoat"
}

RNG_ITEMS = {
    "berserkgene", "scopelens", "quickclaw", "focusband", "brightpowder", "kingsrock"
}

HEALING_MOVES = {
    "rest", "recover", "softboiled", "milkdrink", "synthesis",
    "morningsun", "moonlight", "drainpunch", "megadrain", "gigadrain", "leechlife"
}


ITEM_GROUPS = {
    "leftovers": {"leftovers"},
    "junk": JUNK_ITEMS,
    "status_berry": STATUS_BERRIES,
    "type_boost": TYPE_BOOST_ITEMS,
    "rng": RNG_ITEMS,
}
ITEM_METRICS = [*ITEM_GROUPS, "non_leftovers"]
KNOWN_ITEMS = set().union(*ITEM_GROUPS.values(), {"thickclub", "lightball"})

DEPLOYED_METRICS = ["deployed_turns", "level", "remaining_effective_hp"]
DEPLOYED_SUMMARIES = [
    "deployed_count", "min_level", "underleveled_count", "deployed_underleveled_count",
    "deployed_min_lvl", "active_wmean_lvl", "lead_level", "rem_effective_hp_ratio",
]
TACTIC_MOVES = {
    "rest_used": {"rest"},
    "sleeptalk_used": {"sleeptalk"},
    "phaze_used": {"roar", "whirlwind"},
}
TACTIC_METRICS = ["moves_used", "switches", "eff_power", *TACTIC_MOVES, "spikes_active"]

ALL_FEATURES = [
    # Keep sums when means only divide by TEAM_SIZE (6) or 30 recorded turns.
    # Constants and empirical duplicates below were checked on training data only.
    # BASE STATS

    "p1_sum_hp",
    "p1_sum_atk",
    "p1_sum_def",
    "p1_sum_spa",
    "p1_sum_spd",
    "p1_sum_spe",
    # "p1_mean_hp",   # Duplicate: p1_sum_hp / TEAM_SIZE
    # "p1_mean_atk",  # Duplicate: p1_sum_atk / TEAM_SIZE
    # "p1_mean_def",  # Duplicate: p1_sum_def / TEAM_SIZE
    # "p1_mean_spa",  # Duplicate: p1_sum_spa / TEAM_SIZE
    # "p1_mean_spd",  # Duplicate: p1_sum_spd / TEAM_SIZE
    # "p1_mean_spe",  # Duplicate: p1_sum_spe / TEAM_SIZE
    "p1_min_base_hp",
    "p1_min_base_atk",
    "p1_min_base_def",
    "p1_min_base_spa",
    "p1_min_base_spd",
    "p1_min_base_spe",
    "p1_max_base_hp",
    "p1_max_base_atk",
    "p1_max_base_def",
    "p1_max_base_spa",
    "p1_max_base_spd",
    "p1_max_base_spe",
    "p1_std_base_hp",
    "p1_std_base_atk",
    "p1_std_base_def",
    "p1_std_base_spa",
    "p1_std_base_spd",
    "p1_std_base_spe",
    "p2_sum_hp",
    "p2_sum_atk",
    "p2_sum_def",
    "p2_sum_spa",
    "p2_sum_spd",
    "p2_sum_spe",
    "p2_mean_hp",
    "p2_mean_atk",
    "p2_mean_def",
    "p2_mean_spa",
    "p2_mean_spd",
    "p2_mean_spe",
    "p2_min_base_hp",
    "p2_min_base_atk",
    "p2_min_base_def",
    "p2_min_base_spa",
    "p2_min_base_spd",
    "p2_min_base_spe",
    "p2_max_base_hp",
    "p2_max_base_atk",
    "p2_max_base_def",
    "p2_max_base_spa",
    "p2_max_base_spd",
    "p2_max_base_spe",
    "p2_std_base_hp",
    "p2_std_base_atk",
    "p2_std_base_def",
    "p2_std_base_spa",
    "p2_std_base_spd",
    "p2_std_base_spe",
    "p1_sum_bst",
    # "p1_mean_bst",  # Duplicate: p1_sum_bst / TEAM_SIZE
    "p1_std_base_bst",
    "p2_sum_bst",
    # "p2_mean_bst",  # Duplicate: p2_sum_bst / TEAM_SIZE
    "p2_std_base_bst",
    "diff_sum_hp",
    "diff_sum_atk",
    "diff_sum_def",
    "diff_sum_spa",
    "diff_sum_spd",
    "diff_sum_spe",
    "diff_mean_hp",
    "diff_mean_atk",
    "diff_mean_def",
    "diff_mean_spa",
    "diff_mean_spd",
    "diff_mean_spe",
    "diff_sum_bst",
    # "diff_mean_bst",  # Duplicate: diff_sum_bst / TEAM_SIZE
    "diff_std_base_bst",

    # HP PCT LOST

    "p1_hp_pct_lost_0_10",
    "p1_hp_pct_lost_10_20",
    "p1_hp_pct_lost_20_30",
    "p1_sum_hp_lost",
    # "p1_mean_hp_pct_lost",  # Duplicate: p1_sum_hp_lost / TEAM_SIZE
    "p1_std_hp_pct_lost",
    "p2_hp_pct_lost_0_10",
    "p2_hp_pct_lost_10_20",
    "p2_hp_pct_lost_20_30",
    "p2_sum_hp_lost",
    # "p2_mean_hp_pct_lost",  # Duplicate: p2_sum_hp_lost / TEAM_SIZE
    "p2_std_hp_pct_lost",
    "diff_hp_pct_lost_0_10",
    "diff_hp_pct_lost_10_20",
    "diff_hp_pct_lost_20_30",
    "diff_sum_hp_pct_lost",
    # "diff_mean_hp_pct_lost",  # Duplicate: diff_sum_hp_pct_lost / TEAM_SIZE
    "diff_std_hp_pct_lost",

    # TEAM TYPE

    # "p1_known_type_count",  # Constant in current training data: 6
    "p1_type_diversity",
    "p1_average_weakness_severity",
    "p1_shared_weakness_type_count",
    "p2_known_type_count",
    "p2_type_diversity",
    "p2_average_weakness_severity",
    "p2_shared_weakness_type_count",
    "diff_type_diversity",
    "diff_average_weakness_severity",
    "diff_shared_weakness_type_count",

    # MULTIPLIERS

    "p1_0_mult",
    "p1_0.25_mult",
    "p1_0.5_mult", 
    "p1_1_mult",
    "p1_2_mult",
    "p1_4_mult",
    "p2_0_mult",
    "p2_0.25_mult",
    "p2_0.5_mult", 
    "p2_1_mult",
    "p2_2_mult",
    "p2_4_mult",
    "diff_0_mult",
    "diff_0.25_mult",
    "diff_0.5_mult",
    "diff_1_mult",
    "diff_2_mult",
    "diff_4_mult",
    "p1_count_mult",
    "p2_count_mult",
    "p1_sum_mult",
    "p2_sum_mult",
    "p1_mean_mult",
    "p2_mean_mult",
    "p1_min_mult",
    "p2_min_mult",
    "p1_max_mult",
    "p2_max_mult",
    "p1_std_mult",
    "p2_std_mult",
    "diff_count_mult",
    "diff_sum_mult",
    "diff_mean_mult",
    "diff_min_mult",
    "diff_max_mult",
    "diff_std_mult",

    # STATUSES
    
    "p1_slp",
    "p1_frz",
    "p1_tox",
    "p1_psn",
    "p1_par",
    "p1_brn",
    "p2_slp",
    "p2_frz",
    "p2_tox",
    "p2_psn",
    "p2_par",
    "p2_brn",
    "p1_sum_statuses",
    # "p1_status_frac_slp",  # Duplicate: p1_slp / 30
    # "p1_status_frac_frz",  # Duplicate: p1_frz / 30
    # "p1_status_frac_tox",  # Duplicate: p1_tox / 30
    # "p1_status_frac_psn",  # Duplicate: p1_psn / 30
    # "p1_status_frac_par",  # Duplicate: p1_par / 30
    # "p1_status_frac_brn",  # Duplicate: p1_brn / 30
    # "p1_mean_statuses",    # Duplicate: p1_sum_statuses / 30
    "p2_sum_statuses",
    # "p2_status_frac_slp",  # Duplicate: p2_slp / 30
    # "p2_status_frac_frz",  # Duplicate: p2_frz / 30
    # "p2_status_frac_tox",  # Duplicate: p2_tox / 30
    # "p2_status_frac_psn",  # Duplicate: p2_psn / 30
    # "p2_status_frac_par",  # Duplicate: p2_par / 30
    # "p2_status_frac_brn",  # Duplicate: p2_brn / 30
    # "p2_mean_statuses",    # Duplicate: p2_sum_statuses / 30
    "diff_slp",
    "diff_frz",
    "diff_tox",
    "diff_psn",
    "diff_par",
    "diff_brn",
    # "diff_status_frac_slp",  # Duplicate: diff_slp / 30
    # "diff_status_frac_frz",  # Duplicate: diff_frz / 30
    # "diff_status_frac_tox",  # Duplicate: diff_tox / 30
    # "diff_status_frac_psn",  # Duplicate: diff_psn / 30
    # "diff_status_frac_par",  # Duplicate: diff_par / 30
    # "diff_status_frac_brn",  # Duplicate: diff_brn / 30
    "diff_sum_statuses",
    # "diff_mean_statuses",    # Duplicate: diff_sum_statuses / 30

    # BOOSTS
    
    "p1_atk_boost",
    "p1_def_boost",
    "p1_spa_boost",
    "p1_spd_boost",
    "p1_spe_boost",
    "p2_atk_boost",
    "p2_def_boost",
    "p2_spa_boost",
    "p2_spd_boost",
    "p2_spe_boost",
    "p1_sum_boosts",
    "p2_sum_boosts",
    # "p1_mean_boosts",  # Duplicate: p1_sum_boosts / 30
    # "p2_mean_boosts",  # Duplicate: p2_sum_boosts / 30
    "diff_atk_boost",
    "diff_def_boost",
    "diff_spa_boost",
    "diff_spd_boost",
    "diff_spe_boost",
    "diff_sum_boosts",
    # "diff_mean_boosts",  # Duplicate: diff_sum_boosts / 30

    # EFFECTS

    "p1_sum_good_effects",
    # "p1_beatup_effect",       # 0.0067%
    # "p1_bide_effect",         # 0.0333%
    # "p1_destinybond_effect",  # 1.3000%
    # "p1_endure_effect",       # 0.1333%
    # "p1_focusband_effect",    # 0.0400%
    # "p1_focusenergy_effect",  # 0.0533%
    # "p1_lockon_effect",       # 0.0000%
    # "p1_magnitude_effect",    # 0.0533%
    # "p1_mimic_effect",        # 0.0600%
    # "p1_mindreader_effect",   # 0.0067%
    # "p1_mysteryberry_effect", # 0.0000%
    # "p1_protect_effect",      # 4.1467%
    # "p1_quickclaw_effect",    # 0.0333%
    # "p1_safeguard_effect",    # 0.0000%
    # "p1_substitute_effect",   # 6.8467%
    # "p1_typechange_effect",   # 0.0400%
    # "p1_mean_good_effects",   # Duplicate: p1_sum_good_effects / 30
    "p1_sum_bad_effects",
    # "p1_attract_effect",      # 0.4733%
    # "p1_bind_effect",         # 0.0333%
    # "p1_clamp_effect",        # 0.7267%
    # "p1_confusion_effect",    # 9.8333%
    # "p1_curse_effect",        # 0.0933%
    # "p1_disable_effect",      # 0.0533%
    # "p1_encore_effect",       # 0.7667%
    # "p1_firespin_effect",     # 0.0733%
    # "p1_foresight_effect",    # 0.0000%
    # "p1_futuresight_effect",  # 0.0533%
    # "p1_leechseed_effect",    # 4.5733%
    # "p1_nightmare_effect",    # 0.3933%
    # "p1_perish_effect",       # 3.4800%
    # "p1_pursuit_effect",      # 0.7200%
    # "p1_spite_effect",        # 0.0067%
    # "p1_trapped_effect",      # 5.7667%
    # "p1_whirlpool_effect",    # 0.1933%
    # "p1_wrap_effect",         # 0.1133%
    # "p1_mean_bad_effects",    # Duplicate: p1_sum_bad_effects / 30
    "p1_sum_effects",
    # "p1_mean_effects",        # Duplicate: p1_sum_effects / 30
    "p2_sum_good_effects",
    # "p2_beatup_effect",       # 0.0133%
    # "p2_bide_effect",         # 0.0467%
    # "p2_destinybond_effect",  # 1.1133%
    # "p2_endure_effect",       # 0.2000%
    # "p2_focusband_effect",    # 0.0533%
    # "p2_focusenergy_effect",  # 0.0733%
    # "p2_lockon_effect",       # 0.0133%
    # "p2_magnitude_effect",    # 0.0400%
    # "p2_mimic_effect",        # 0.0400%
    # "p2_mindreader_effect",   # 0.0200%
    # "p2_mysteryberry_effect", # 0.0133%
    # "p2_protect_effect",      # 4.6667%
    # "p2_quickclaw_effect",    # 0.0133%
    # "p2_safeguard_effect",    # 0.0267%
    # "p2_substitute_effect",   # 6.7067%
    # "p2_typechange_effect",   # 0.0133%
    # "p2_mean_good_effects",   # Duplicate: p2_sum_good_effects / 30
    "p2_sum_bad_effects",
    # "p2_attract_effect",      # 0.5067%
    # "p2_bind_effect",         # 0.0333%
    # "p2_clamp_effect",        # 0.7133%
    # "p2_confusion_effect",    # 10.1000%
    # "p2_curse_effect",        # 0.0667%
    # "p2_disable_effect",      # 0.0467%
    # "p2_encore_effect",       # 0.7933%
    # "p2_firespin_effect",     # 0.0733%
    # "p2_foresight_effect",    # 0.0067%
    # "p2_futuresight_effect",  # 0.0133%
    # "p2_leechseed_effect",    # 4.3200%
    # "p2_nightmare_effect",    # 0.5133%
    # "p2_perish_effect",       # 3.2867%
    # "p2_pursuit_effect",      # 0.0000%
    # "p2_spite_effect",        # 0.0133%
    # "p2_trapped_effect",      # 6.0400%
    # "p2_whirlpool_effect",    # 0.2067%
    # "p2_wrap_effect",         # 0.0933%
    # "p2_mean_bad_effects",    # Duplicate: p2_sum_bad_effects / 30
    "p2_sum_effects",
    # "p2_mean_effects",            # Duplicate: p2_sum_effects / 30
    # "diff_beatup_effect",         # 0.0200%
    # "diff_bide_effect",           # 0.0800%
    # "diff_destinybond_effect",    # 2.3867%
    # "diff_endure_effect",         # 0.3333%
    # "diff_focusband_effect",      # 0.0933%
    # "diff_focusenergy_effect",    # 0.1200%
    # "diff_lockon_effect",         # 0.0133%
    # "diff_magnitude_effect",      # 0.0933%
    # "diff_mimic_effect",          # 0.1000%
    # "diff_mindreader_effect",     # 0.0267%
    # "diff_mysteryberry_effect",   # 0.0133%
    # "diff_protect_effect",        # 8.5267%
    # "diff_quickclaw_effect",      # 0.0467%
    # "diff_safeguard_effect",      # 0.0267%
    # "diff_substitute_effect",     # 12.9133%
    # "diff_typechange_effect",     # 0.0400%
    "diff_sum_good_effects",
    # "diff_mean_good_effects",     # Duplicate: diff_sum_good_effects / 30
    # "diff_attract_effect",        # 0.9733%
    # "diff_bind_effect",           # 0.0667%
    # "diff_clamp_effect",          # 1.4267%
    # "diff_confusion_effect",      # 18.4333%
    # "diff_curse_effect",          # 0.1600%
    # "diff_disable_effect",        # 0.1000%
    # "diff_encore_effect",         # 1.5333%
    # "diff_firespin_effect",       # 0.1467%
    # "diff_foresight_effect",      # 0.0067%
    # "diff_futuresight_effect",    # 0.0667%
    # "diff_leechseed_effect",      # 8.7200%
    # "diff_nightmare_effect",      # 0.8933%
    # "diff_perish_effect",         # 4.4400%
    # "diff_pursuit_effect",        # 0.7200%
    # "diff_spite_effect",          # 0.0200%
    # "diff_trapped_effect",        # 11.2133%
    # "diff_whirlpool_effect",      # 0.4000%
    # "diff_wrap_effect",           # 0.2000%
    "diff_sum_bad_effects",
    # "diff_mean_bad_effects",      # Duplicate: diff_sum_bad_effects / 30
    "diff_sum_effects",
    # "diff_mean_effects",          # Duplicate: diff_sum_effects / 30

    # LAST-SEEN SNAPSHOT

    "p1_0_pkm_remaining_hp_pct",
    "p1_0_pkm_remaining_status",
    "p1_1_pkm_remaining_hp_pct",
    "p1_1_pkm_remaining_status",
    "p1_2_pkm_remaining_hp_pct",
    "p1_2_pkm_remaining_status",
    "p1_3_pkm_remaining_hp_pct",
    "p1_3_pkm_remaining_status",
    "p1_4_pkm_remaining_hp_pct",
    "p1_4_pkm_remaining_status",
    "p1_5_pkm_remaining_hp_pct",
    "p1_5_pkm_remaining_status",
    "p2_0_pkm_remaining_hp_pct",
    "p2_0_pkm_remaining_status",
    "p2_1_pkm_remaining_hp_pct",
    "p2_1_pkm_remaining_status",
    "p2_2_pkm_remaining_hp_pct",
    "p2_2_pkm_remaining_status",
    "p2_3_pkm_remaining_hp_pct",
    "p2_3_pkm_remaining_status",
    "p2_4_pkm_remaining_hp_pct",
    "p2_4_pkm_remaining_status",
    "p2_5_pkm_remaining_hp_pct",
    "p2_5_pkm_remaining_status",
    # "p1_sum_remaining_hp_pct",  # Duplicate: TEAM_SIZE - p1_sum_hp_lost
    # "p1_std_remaining_hp_pct",  # Duplicate: p1_std_hp_pct_lost
    # "p2_sum_remaining_hp_pct",  # Duplicate: TEAM_SIZE - p2_sum_hp_lost
    # "p2_std_remaining_hp_pct",  # Duplicate: p2_std_hp_pct_lost
    # "diff_sum_remaining_hp_pct",  # Duplicate: diff_sum_hp_pct_lost
    # "diff_std_remaining_hp_pct",  # Duplicate: -diff_std_hp_pct_lost
    "p1_remaining_slp",
    "p1_remaining_frz",
    "p1_remaining_tox",
    "p1_remaining_psn",
    "p1_remaining_par",
    "p1_remaining_brn",
    "p2_remaining_slp",
    "p2_remaining_frz",
    "p2_remaining_tox",
    "p2_remaining_psn",
    "p2_remaining_par",
    "p2_remaining_brn",
    "diff_remaining_slp",
    "diff_remaining_frz",
    "diff_remaining_tox",
    "diff_remaining_psn",
    "diff_remaining_par",
    "diff_remaining_brn",
    "p1_remaining_atk_boost",
    "p1_remaining_def_boost",
    "p1_remaining_spa_boost",
    "p1_remaining_spd_boost",
    "p1_remaining_spe_boost",
    "p2_remaining_atk_boost",
    "p2_remaining_def_boost",
    "p2_remaining_spa_boost",
    "p2_remaining_spd_boost",
    "p2_remaining_spe_boost",
    "diff_remaining_atk_boost",
    "diff_remaining_def_boost",
    "diff_remaining_spa_boost",
    "diff_remaining_spd_boost",
    "diff_remaining_spe_boost",
    "p1_sum_remaining_statuses",
    "p1_sum_remaining_boosts",
    "p1_sum_remaining_good_effects",
    "p1_sum_remaining_bad_effects",
    "p1_sum_remaining_effects",
    "p2_sum_remaining_statuses",
    "p2_sum_remaining_boosts",
    "p2_sum_remaining_good_effects",
    "p2_sum_remaining_bad_effects",
    "p2_sum_remaining_effects",
    "diff_sum_remaining_statuses",
    "diff_sum_remaining_boosts",
    "diff_sum_remaining_good_effects",
    "diff_sum_remaining_bad_effects",
    "diff_sum_remaining_effects",

    # FAINT EVIDENCE

    "p1_0_pkm_confirmed_kos",
    # "p1_0_pkm_estimated_true_kos",  # Duplicate: p1_0_pkm_confirmed_kos
    "p1_1_pkm_confirmed_kos",
    # "p1_1_pkm_estimated_true_kos",  # Duplicate: p1_1_pkm_confirmed_kos
    "p1_2_pkm_confirmed_kos",
    # "p1_2_pkm_estimated_true_kos",  # Duplicate: p1_2_pkm_confirmed_kos
    "p1_3_pkm_confirmed_kos",
    # "p1_3_pkm_estimated_true_kos",  # Duplicate: p1_3_pkm_confirmed_kos
    "p1_4_pkm_confirmed_kos",
    # "p1_4_pkm_estimated_true_kos",  # Duplicate: p1_4_pkm_confirmed_kos
    "p1_5_pkm_confirmed_kos",
    # "p1_5_pkm_estimated_true_kos",  # Duplicate: p1_5_pkm_confirmed_kos
    "p2_0_pkm_confirmed_kos",
    "p2_0_pkm_estimated_true_kos",
    "p2_1_pkm_confirmed_kos",
    "p2_1_pkm_estimated_true_kos",
    "p2_2_pkm_confirmed_kos",
    "p2_2_pkm_estimated_true_kos",
    "p2_3_pkm_confirmed_kos",
    "p2_3_pkm_estimated_true_kos",
    "p2_4_pkm_confirmed_kos",
    "p2_4_pkm_estimated_true_kos",
    "p2_5_pkm_confirmed_kos",
    "p2_5_pkm_estimated_true_kos",
    "p1_sum_confirmed_kos",
    # "p1_sum_estimated_true_kos",  # Duplicate: p1_sum_confirmed_kos
    # "p1_mean_confirmed_kos",  # Duplicate: p1_sum_confirmed_kos / TEAM_SIZE
    # "p1_mean_estimated_true_kos",  # Duplicate: (p1_sum_confirmed_kos) / TEAM_SIZE
    "p1_std_confirmed_kos",
    # "p1_std_estimated_true_kos",  # Duplicate: p1_std_confirmed_kos
    "p2_sum_confirmed_kos",
    "p2_sum_estimated_true_kos",
    # "p2_mean_confirmed_kos",  # Duplicate: p2_sum_confirmed_kos / TEAM_SIZE
    # "p2_mean_estimated_true_kos",  # Duplicate: p2_sum_estimated_true_kos / TEAM_SIZE
    "p2_std_confirmed_kos",
    "p2_std_estimated_true_kos",
    "diff_sum_confirmed_kos",
    "diff_sum_estimated_true_kos",
    # "diff_mean_confirmed_kos",  # Duplicate: diff_sum_confirmed_kos / TEAM_SIZE
    # "diff_mean_estimated_true_kos",  # Duplicate: diff_sum_estimated_true_kos / TEAM_SIZE
    "diff_std_confirmed_kos",
    "diff_std_estimated_true_kos",
    # "p1_confirmed_kos",  # Duplicate: p1_sum_confirmed_kos
    # "p1_estimated_true_kos",  # Duplicate: p1_sum_confirmed_kos
    # "p1_estimated_pokemon_alive",  # Duplicate: TEAM_SIZE - ((p1_sum_confirmed_kos))
    # "p2_confirmed_kos",  # Duplicate: p2_sum_confirmed_kos
    # "p2_estimated_true_kos",  # Duplicate: p2_sum_estimated_true_kos
    # "p2_estimated_pokemon_alive",  # Duplicate: TEAM_SIZE - (p2_sum_estimated_true_kos)
    # "diff_confirmed_kos",  # Duplicate: diff_sum_confirmed_kos
    # "diff_estimated_true_kos",  # Duplicate: diff_sum_estimated_true_kos
    # "diff_estimated_pokemon_alive",  # Duplicate: diff_sum_estimated_true_kos
    # "p1_fnt",  # Duplicate: p1_sum_confirmed_kos
    # "p1_known_remaining_hp_sum",  # Duplicate: TEAM_SIZE - p1_sum_hp_lost
    # "p2_fnt",  # Duplicate: p2_sum_confirmed_kos
    # "p2_known_remaining_hp_sum",  # Duplicate: TEAM_SIZE - p2_sum_hp_lost

    # HELD ITEMS (P2 VALUES ARE OBSERVED EVIDENCE)

    "p1_0_pkm_item",
    "p1_1_pkm_item",
    "p1_2_pkm_item",
    "p1_3_pkm_item",
    "p1_4_pkm_item",
    "p1_5_pkm_item",
    "p2_0_pkm_item",
    "p2_1_pkm_item",
    "p2_2_pkm_item",
    "p2_3_pkm_item",
    "p2_4_pkm_item",
    "p2_5_pkm_item",
    "p1_0_pkm_item_leftovers",
    "p1_0_pkm_item_junk",
    "p1_0_pkm_item_status_berry",
    "p1_0_pkm_item_type_boost",
    "p1_0_pkm_item_rng",
    # "p1_0_pkm_item_non_leftovers",  # Duplicate: 1 - p1_0_pkm_item_leftovers
    "p1_1_pkm_item_leftovers",
    "p1_1_pkm_item_junk",
    "p1_1_pkm_item_status_berry",
    "p1_1_pkm_item_type_boost",
    "p1_1_pkm_item_rng",
    # "p1_1_pkm_item_non_leftovers",  # Duplicate: 1 - p1_1_pkm_item_leftovers
    "p1_2_pkm_item_leftovers",
    "p1_2_pkm_item_junk",
    "p1_2_pkm_item_status_berry",
    "p1_2_pkm_item_type_boost",
    "p1_2_pkm_item_rng",
    # "p1_2_pkm_item_non_leftovers",  # Duplicate: 1 - p1_2_pkm_item_leftovers
    "p1_3_pkm_item_leftovers",
    "p1_3_pkm_item_junk",
    "p1_3_pkm_item_status_berry",
    "p1_3_pkm_item_type_boost",
    "p1_3_pkm_item_rng",
    # "p1_3_pkm_item_non_leftovers",  # Duplicate: 1 - p1_3_pkm_item_leftovers
    "p1_4_pkm_item_leftovers",
    "p1_4_pkm_item_junk",
    "p1_4_pkm_item_status_berry",
    "p1_4_pkm_item_type_boost",
    "p1_4_pkm_item_rng",
    # "p1_4_pkm_item_non_leftovers",  # Duplicate: 1 - p1_4_pkm_item_leftovers
    "p1_5_pkm_item_leftovers",
    "p1_5_pkm_item_junk",
    "p1_5_pkm_item_status_berry",
    "p1_5_pkm_item_type_boost",
    "p1_5_pkm_item_rng",
    # "p1_5_pkm_item_non_leftovers",  # Duplicate: 1 - p1_5_pkm_item_leftovers
    "p2_0_pkm_item_leftovers",
    # "p2_0_pkm_item_junk",  # Constant in training: 0
    # "p2_0_pkm_item_status_berry",  # Constant in training: 0
    # "p2_0_pkm_item_type_boost",  # Constant in training: 0
    # "p2_0_pkm_item_rng",  # Constant in training: 0
    # "p2_0_pkm_item_non_leftovers",  # Constant in training: 0
    "p2_1_pkm_item_leftovers",
    # "p2_1_pkm_item_junk",  # Constant in training: 0
    # "p2_1_pkm_item_status_berry",  # Constant in training: 0
    # "p2_1_pkm_item_type_boost",  # Constant in training: 0
    "p2_1_pkm_item_rng",
    # "p2_1_pkm_item_non_leftovers",  # Duplicate: p2_1_pkm_item_rng (same detected Pokemon in training)
    "p2_2_pkm_item_leftovers",
    # "p2_2_pkm_item_junk",  # Constant in training: 0
    # "p2_2_pkm_item_status_berry",  # Constant in training: 0
    # "p2_2_pkm_item_type_boost",  # Constant in training: 0
    "p2_2_pkm_item_rng",
    # "p2_2_pkm_item_non_leftovers",  # Duplicate: p2_2_pkm_item_rng (same detected Pokemon in training)
    "p2_3_pkm_item_leftovers",
    # "p2_3_pkm_item_junk",  # Constant in training: 0
    # "p2_3_pkm_item_status_berry",  # Constant in training: 0
    # "p2_3_pkm_item_type_boost",  # Constant in training: 0
    "p2_3_pkm_item_rng",
    # "p2_3_pkm_item_non_leftovers",  # Duplicate: p2_3_pkm_item_rng (same detected Pokemon in training)
    "p2_4_pkm_item_leftovers",
    # "p2_4_pkm_item_junk",  # Constant in training: 0
    # "p2_4_pkm_item_status_berry",  # Constant in training: 0
    # "p2_4_pkm_item_type_boost",  # Constant in training: 0
    "p2_4_pkm_item_rng",
    # "p2_4_pkm_item_non_leftovers",  # Duplicate: p2_4_pkm_item_rng (same detected Pokemon in training)
    "p2_5_pkm_item_leftovers",
    # "p2_5_pkm_item_junk",  # Constant in training: 0
    # "p2_5_pkm_item_status_berry",  # Constant in training: 0
    # "p2_5_pkm_item_type_boost",  # Constant in training: 0
    "p2_5_pkm_item_rng",
    # "p2_5_pkm_item_non_leftovers",  # Duplicate: p2_5_pkm_item_rng (same detected Pokemon in training)
    "p1_sum_item_leftovers",
    "p1_sum_item_junk",
    "p1_sum_item_status_berry",
    "p1_sum_item_type_boost",
    "p1_sum_item_rng",
    # "p1_sum_item_non_leftovers",  # Duplicate: TEAM_SIZE - p1_sum_item_leftovers
    # "p1_mean_item_leftovers",  # Duplicate: p1_sum_item_leftovers / TEAM_SIZE
    # "p1_mean_item_junk",  # Duplicate: p1_sum_item_junk / TEAM_SIZE
    # "p1_mean_item_status_berry",  # Duplicate: p1_sum_item_status_berry / TEAM_SIZE
    # "p1_mean_item_type_boost",  # Duplicate: p1_sum_item_type_boost / TEAM_SIZE
    # "p1_mean_item_rng",  # Duplicate: p1_sum_item_rng / TEAM_SIZE
    # "p1_mean_item_non_leftovers",  # Duplicate: (TEAM_SIZE - p1_sum_item_leftovers) / TEAM_SIZE
    "p1_std_item_leftovers",
    "p1_std_item_junk",
    "p1_std_item_status_berry",
    "p1_std_item_type_boost",
    "p1_std_item_rng",
    # "p1_std_item_non_leftovers",  # Duplicate: p1_std_item_leftovers
    "p2_sum_item_leftovers",
    # "p2_sum_item_junk",  # Constant in training: 0
    # "p2_sum_item_status_berry",  # Constant in training: 0
    # "p2_sum_item_type_boost",  # Constant in training: 0
    "p2_sum_item_rng",
    # "p2_sum_item_non_leftovers",  # Duplicate: p2_sum_item_rng (same detected Pokemon in training)
    # "p2_mean_item_leftovers",  # Duplicate: p2_sum_item_leftovers / TEAM_SIZE
    # "p2_mean_item_junk",  # Constant in training: 0
    # "p2_mean_item_status_berry",  # Constant in training: 0
    # "p2_mean_item_type_boost",  # Constant in training: 0
    # "p2_mean_item_rng",  # Duplicate: p2_sum_item_rng / TEAM_SIZE
    # "p2_mean_item_non_leftovers",  # Duplicate: (p2_sum_item_rng) / TEAM_SIZE (same detected Pokemon in training)
    "p2_std_item_leftovers",
    # "p2_std_item_junk",  # Constant in training: 0
    # "p2_std_item_status_berry",  # Constant in training: 0
    # "p2_std_item_type_boost",  # Constant in training: 0
    # "p2_std_item_rng",  # Duplicate: sqrt(TEAM_SIZE - 1) * p2_sum_item_rng / TEAM_SIZE (count is 0 or 1 in training)
    # "p2_std_item_non_leftovers",  # Duplicate: sqrt(TEAM_SIZE - 1) * p2_sum_item_rng / TEAM_SIZE (count is 0 or 1 in training; same detected Pokemon in training)
    "diff_sum_item_leftovers",
    # "diff_sum_item_junk",  # Duplicate: p1_sum_item_junk (P2 value is 0 in training)
    # "diff_sum_item_status_berry",  # Duplicate: p1_sum_item_status_berry (P2 value is 0 in training)
    # "diff_sum_item_type_boost",  # Duplicate: p1_sum_item_type_boost (P2 value is 0 in training)
    "diff_sum_item_rng",
    "diff_sum_item_non_leftovers",
    # "diff_mean_item_leftovers",  # Duplicate: diff_sum_item_leftovers / TEAM_SIZE
    # "diff_mean_item_junk",  # Duplicate: (p1_sum_item_junk) / TEAM_SIZE (P2 value is 0 in training)
    # "diff_mean_item_status_berry",  # Duplicate: (p1_sum_item_status_berry) / TEAM_SIZE (P2 value is 0 in training)
    # "diff_mean_item_type_boost",  # Duplicate: (p1_sum_item_type_boost) / TEAM_SIZE (P2 value is 0 in training)
    # "diff_mean_item_rng",  # Duplicate: diff_sum_item_rng / TEAM_SIZE
    # "diff_mean_item_non_leftovers",  # Duplicate: diff_sum_item_non_leftovers / TEAM_SIZE
    "diff_std_item_leftovers",
    # "diff_std_item_junk",  # Duplicate: p1_std_item_junk (P2 value is 0 in training)
    # "diff_std_item_status_berry",  # Duplicate: p1_std_item_status_berry (P2 value is 0 in training)
    # "diff_std_item_type_boost",  # Duplicate: p1_std_item_type_boost (P2 value is 0 in training)
    "diff_std_item_rng",
    "diff_std_item_non_leftovers",
    # "p1_item_leftovers_count",  # Duplicate: p1_sum_item_leftovers
    # "p1_item_junk_count",  # Duplicate: p1_sum_item_junk
    # "p1_item_status_berry_count",  # Duplicate: p1_sum_item_status_berry
    # "p1_item_type_boost_count",  # Duplicate: p1_sum_item_type_boost
    # "p1_item_rng_count",  # Duplicate: p1_sum_item_rng
    # "p1_item_non_leftovers_count",  # Duplicate: TEAM_SIZE - p1_sum_item_leftovers
    # "p2_item_leftovers_count",  # Duplicate: p2_sum_item_leftovers
    # "p2_item_junk_count",  # Constant in training: 0
    # "p2_item_status_berry_count",  # Constant in training: 0
    # "p2_item_type_boost_count",  # Constant in training: 0
    # "p2_item_rng_count",  # Duplicate: p2_sum_item_rng
    # "p2_item_non_leftovers_count",  # Duplicate: p2_sum_item_rng (same detected Pokemon in training)
    "p1_item_has_all_leftovers",
    "p1_item_unique_count",
    # "p1_item_non_leftovers",  # Duplicate: TEAM_SIZE - p1_sum_item_leftovers
    "p2_item_has_all_leftovers",
    "p2_item_unique_count",
    # "p2_item_non_leftovers",  # Duplicate: p2_sum_item_rng (same detected Pokemon in training)
    "p1_has_thickclub_marowak",
    "p1_has_lightball_pikachu",
    # "p2_detected_leftovers_count",  # Duplicate: p2_sum_item_leftovers
    # "p2_detected_rng_count",  # Duplicate: p2_sum_item_rng
    # "diff_leftovers_count",  # Duplicate: diff_sum_item_leftovers
    # "diff_junk_items_count",  # Duplicate: p1_sum_item_junk (P2 value is 0 in training)

    # DEPLOYED POKEMON AND LEVELS

    "p1_0_pkm_deployed_turns",
    "p1_0_pkm_level",
    "p1_0_pkm_remaining_effective_hp",
    "p1_1_pkm_deployed_turns",
    "p1_1_pkm_level",
    "p1_1_pkm_remaining_effective_hp",
    "p1_2_pkm_deployed_turns",
    "p1_2_pkm_level",
    "p1_2_pkm_remaining_effective_hp",
    "p1_3_pkm_deployed_turns",
    "p1_3_pkm_level",
    "p1_3_pkm_remaining_effective_hp",
    "p1_4_pkm_deployed_turns",
    "p1_4_pkm_level",
    "p1_4_pkm_remaining_effective_hp",
    "p1_5_pkm_deployed_turns",
    "p1_5_pkm_level",
    "p1_5_pkm_remaining_effective_hp",
    "p2_0_pkm_deployed_turns",
    "p2_0_pkm_level",
    "p2_0_pkm_remaining_effective_hp",
    "p2_1_pkm_deployed_turns",
    "p2_1_pkm_level",
    "p2_1_pkm_remaining_effective_hp",
    "p2_2_pkm_deployed_turns",
    "p2_2_pkm_level",
    "p2_2_pkm_remaining_effective_hp",
    "p2_3_pkm_deployed_turns",
    "p2_3_pkm_level",
    "p2_3_pkm_remaining_effective_hp",
    "p2_4_pkm_deployed_turns",
    "p2_4_pkm_level",
    "p2_4_pkm_remaining_effective_hp",
    "p2_5_pkm_deployed_turns",
    "p2_5_pkm_level",
    "p2_5_pkm_remaining_effective_hp",
    # "p1_sum_deployed_turns",  # Constant in training: 30
    "p1_sum_level",
    "p1_sum_remaining_effective_hp",
    # "p1_mean_deployed_turns",  # Constant in training: 5
    # "p1_mean_level",  # Duplicate: p1_sum_level / TEAM_SIZE
    # "p1_mean_remaining_effective_hp",  # Duplicate: p1_sum_remaining_effective_hp / TEAM_SIZE
    "p1_std_deployed_turns",
    "p1_std_level",
    "p1_std_remaining_effective_hp",
    # "p2_sum_deployed_turns",  # Constant in training: 30
    # "p2_sum_level",  # Duplicate: 100 * (TEAM_SIZE - 1) + p2_lead_level (other P2 levels assumed 100)
    "p2_sum_remaining_effective_hp",
    # "p2_mean_deployed_turns",  # Constant in training: 5
    # "p2_mean_level",  # Duplicate: (100 * (TEAM_SIZE - 1) + p2_lead_level) / TEAM_SIZE (other P2 levels assumed 100)
    # "p2_mean_remaining_effective_hp",  # Duplicate: p2_sum_remaining_effective_hp / TEAM_SIZE
    "p2_std_deployed_turns",
    # "p2_std_level",  # Duplicate: sqrt(TEAM_SIZE - 1) * (100 - p2_lead_level) / TEAM_SIZE (other P2 levels assumed 100)
    "p2_std_remaining_effective_hp",
    # "diff_sum_deployed_turns",  # Constant in training: 0
    "diff_sum_level",
    "diff_sum_remaining_effective_hp",
    # "diff_mean_deployed_turns",  # Constant in training: 0
    # "diff_mean_level",  # Duplicate: diff_sum_level / TEAM_SIZE
    # "diff_mean_remaining_effective_hp",  # Duplicate: diff_sum_remaining_effective_hp / TEAM_SIZE
    "diff_std_deployed_turns",
    "diff_std_level",
    "diff_std_remaining_effective_hp",
    "p1_deployed_count",
    "p1_min_level",
    "p1_underleveled_count",
    "p1_deployed_underleveled_count",
    "p1_deployed_min_lvl",
    "p1_active_wmean_lvl",
    "p1_lead_level",
    "p1_rem_effective_hp_ratio",
    "p2_deployed_count",
    # "p2_min_level",  # Duplicate: p2_lead_level (other P2 levels assumed 100)
    "p2_underleveled_count",
    "p2_deployed_underleveled_count",
    "p2_deployed_min_lvl",
    "p2_active_wmean_lvl",
    "p2_lead_level",
    "p2_rem_effective_hp_ratio",
    "diff_deployed_count",
    "diff_min_level",
    "diff_underleveled_count",
    "diff_deployed_underleveled_count",
    "diff_deployed_min_lvl",
    "diff_active_wmean_lvl",
    "diff_lead_level",
    "diff_rem_effective_hp_ratio",

    # TACTICAL TEMPO, POWER AND HAZARDS

    "p1_sum_moves_used",
    "p1_sum_switches",
    "p1_sum_eff_power",
    "p1_sum_rest_used",
    "p1_sum_sleeptalk_used",
    "p1_sum_phaze_used",
    "p1_sum_spikes_active",
    # "p1_mean_moves_used",  # Duplicate: p1_sum_moves_used / 30
    # "p1_mean_switches",  # Duplicate: p1_sum_switches / 30
    # "p1_mean_eff_power",  # Duplicate: p1_sum_eff_power / 30
    # "p1_mean_rest_used",  # Duplicate: p1_sum_rest_used / 30
    # "p1_mean_sleeptalk_used",  # Duplicate: p1_sum_sleeptalk_used / 30
    # "p1_mean_phaze_used",  # Duplicate: p1_sum_phaze_used / 30
    # "p1_mean_spikes_active",  # Duplicate: p1_sum_spikes_active / 30
    "p1_std_moves_used",
    "p1_std_switches",
    "p1_std_eff_power",
    "p1_std_rest_used",
    "p1_std_sleeptalk_used",
    "p1_std_phaze_used",
    "p1_std_spikes_active",
    "p2_sum_moves_used",
    "p2_sum_switches",
    "p2_sum_eff_power",
    "p2_sum_rest_used",
    "p2_sum_sleeptalk_used",
    "p2_sum_phaze_used",
    "p2_sum_spikes_active",
    # "p2_mean_moves_used",  # Duplicate: p2_sum_moves_used / 30
    # "p2_mean_switches",  # Duplicate: p2_sum_switches / 30
    # "p2_mean_eff_power",  # Duplicate: p2_sum_eff_power / 30
    # "p2_mean_rest_used",  # Duplicate: p2_sum_rest_used / 30
    # "p2_mean_sleeptalk_used",  # Duplicate: p2_sum_sleeptalk_used / 30
    # "p2_mean_phaze_used",  # Duplicate: p2_sum_phaze_used / 30
    # "p2_mean_spikes_active",  # Duplicate: p2_sum_spikes_active / 30
    "p2_std_moves_used",
    "p2_std_switches",
    "p2_std_eff_power",
    "p2_std_rest_used",
    "p2_std_sleeptalk_used",
    "p2_std_phaze_used",
    "p2_std_spikes_active",
    "diff_sum_moves_used",
    "diff_sum_switches",
    "diff_sum_eff_power",
    "diff_sum_rest_used",
    "diff_sum_sleeptalk_used",
    "diff_sum_phaze_used",
    "diff_sum_spikes_active",
    # "diff_mean_moves_used",  # Duplicate: diff_sum_moves_used / 30
    # "diff_mean_switches",  # Duplicate: diff_sum_switches / 30
    # "diff_mean_eff_power",  # Duplicate: diff_sum_eff_power / 30
    # "diff_mean_rest_used",  # Duplicate: diff_sum_rest_used / 30
    # "diff_mean_sleeptalk_used",  # Duplicate: diff_sum_sleeptalk_used / 30
    # "diff_mean_phaze_used",  # Duplicate: diff_sum_phaze_used / 30
    # "diff_mean_spikes_active",  # Duplicate: diff_sum_spikes_active / 30
    "diff_std_moves_used",
    "diff_std_switches",
    "diff_std_eff_power",
    "diff_std_rest_used",
    "diff_std_sleeptalk_used",
    "diff_std_phaze_used",
    "diff_std_spikes_active",
    # "p1_moves_used",  # Duplicate: p1_sum_moves_used / 30
    # "p1_switches",  # Duplicate: p1_sum_switches / 30
    # "p1_eff_power_mean",  # Duplicate: p1_sum_eff_power / 30
    # "p1_rest_used",  # Duplicate: p1_sum_rest_used
    # "p1_sleeptalk_used",  # Duplicate: p1_sum_sleeptalk_used
    # "p1_phaze_used",  # Duplicate: p1_sum_phaze_used
    "p1_spikes_active",
    # "p2_moves_used",  # Duplicate: p2_sum_moves_used / 30
    # "p2_switches",  # Duplicate: p2_sum_switches / 30
    # "p2_eff_power_mean",  # Duplicate: p2_sum_eff_power / 30
    # "p2_rest_used",  # Duplicate: p2_sum_rest_used
    # "p2_sleeptalk_used",  # Duplicate: p2_sum_sleeptalk_used
    # "p2_phaze_used",  # Duplicate: p2_sum_phaze_used
    "p2_spikes_active",
    # "diff_moves_used",  # Duplicate: diff_sum_moves_used / 30
    # "diff_switches",  # Duplicate: diff_sum_switches / 30
    # "diff_eff_power_mean",  # Duplicate: diff_sum_eff_power / 30
    # "diff_rest_used",  # Duplicate: diff_sum_rest_used
    # "diff_sleeptalk_used",  # Duplicate: diff_sum_sleeptalk_used
    # "diff_phaze_used",  # Duplicate: diff_sum_phaze_used
    "diff_spikes_active",
]

# --- FEATURE SELECTION ---
# Selection searches ALL_FEATURES; every listed column must be extracted.
# It takes priority over the manual ALLOWED_FEATURES filter.
USE_FEATURE_SELECTION = True
USE_ALLOWED_FEATURES = False
ALLOWED_FEATURES = []  # Paste a selected subset here for later manual runs.

# Shared settings for both runs.
SELECTION_MIN_FEATURES = 1
SELECTION_VALIDATION_FRACTION = 0.25

# Run 1: light exploration of all available ALL_FEATURES; use the settings below.
# Halving tests about 8 sizes for 135 features, 10 for 700, per search.
# Prefer fewer features within 0.2 percentage points of the best inner accuracy.
SELECTION_DROP_FRACTION = 0.5
SELECTION_MAX_ITERATIONS = 500  # Cheaper models for the search only.
SELECTION_SHAP_SAMPLE_SIZE = 512
SELECTION_ACCURACY_TOLERANCE = 0.002

# Run 2: deep, expensive refinement; first reduce ALL_FEATURES to ~20-40
# candidates from run 1 (winner plus promising alternatives). Extraction stays unchanged.
# USE_ALLOWED_FEATURES does not restrict this search; edit ALL_FEATURES itself.
# Suggested overrides; keep the other settings above:
# SELECTION_DROP_FRACTION = 0.1
# SELECTION_MAX_ITERATIONS = 2000
# SELECTION_SHAP_SAMPLE_SIZE = 2048
# SELECTION_ACCURACY_TOLERANCE = 0.001  # Within 0.1 percentage points of the best inner score.


# --- CATBOOST AND VALIDATION ---
RANDOM_STATE = 42
K_FOLDS = 5
N_REPEATS = 3
EARLY_STOPPING_ROUNDS = 100
EARLY_STOPPING_FRACTION = 0.15
DECISION_THRESHOLD = 0.5

# String/category columns are detected automatically. Add integer-coded
# categorical columns here when extending the feature lists.
CATEGORICAL_FEATURES = []

# Starting settings, not tuned results. Early stopping selects the tree count
# using Logloss; compare feature experiments using validation accuracy.
CATBOOST_PARAMS = {
    "iterations": 2000,
    "depth": 6,
    "learning_rate": 0.03,
    "l2_leaf_reg": 5.0,
    "loss_function": "Logloss",
    "eval_metric": "Logloss",
    "task_type": "CPU",
    "thread_count": -1,
    "allow_writing_files": False,
    "verbose": False,
}

## 1. Loading Data

In [ ]:
# --- LOAD DATA ---
def load_data(file_path: str) -> list:
    """
    Loads the whole `file_path` and stores it as a list of battles
    """

    print(f"Loading data from '{file_path}'...")
    data = []
    try:
        with open(file_path, "r") as f:
            for line in f:
                data.append(json.loads(line))
        print(f"SUCCESS: loaded {len(data)} battles.")
        
    except FileNotFoundError:
        print(f"ERROR: could not find the file at '{file_path}'.")

    return data


# Loading data
train_data = load_data(TRAIN_FILE_PATH)
test_data  = load_data(TEST_FILE_PATH)

### Pokemon Stats & Moves Type

In [ ]:
# --- POKEMON TYPE MAP ---
def create_stats_map(train_data: list, test_data: list, file_path: None | str = STATS_MAP_PATH) -> dict:
    """
    Build a species stats/type lookup from supplied battle details
    """

    print("Creating Pokémon stats lookup...")
    stats_map = {}
    for data in (train_data, test_data):
        for battle in data:

            for pokemon in battle["p1_team_details"]:
                if pokemon["name"] not in stats_map:
                    stats_map[pokemon["name"]] = {"base_stats": {}, "types": []}
                stats_map[pokemon["name"]]["base_stats"] = {
                    "base_hp": pokemon["base_hp"],
                    "base_atk": pokemon["base_atk"],
                    "base_def": pokemon["base_def"],
                    "base_spa": pokemon["base_spa"],
                    "base_spd": pokemon["base_spd"],
                    "base_spe": pokemon["base_spe"],
                }
                stats_map[pokemon["name"]]["types"] = pokemon["types"]

            if battle["p2_lead_details"]["name"] not in stats_map:
                stats_map[battle["p2_lead_details"]["name"]] = {"base_stats": {}, "types": []}
            lead = battle["p2_lead_details"]
            stats_map[lead["name"]]["base_stats"] = {
                "base_hp": lead["base_hp"],
                "base_atk": lead["base_atk"],
                "base_def": lead["base_def"],
                "base_spa": lead["base_spa"],
                "base_spd": lead["base_spd"],
                "base_spe": lead["base_spe"],
            }
            stats_map[lead["name"]]["types"] = lead["types"]

    if DEBUG:
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(stats_map, f, indent=4)

    print(f"SUCCESS: found {len(stats_map)} pokemon.")
    return stats_map


# --- MOVES TYPE JSON ---
def normalize_type_chart(type_chart: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize type-chart column names and type values in place
    """

    # Clear columns headers
    type_chart.columns = type_chart.columns.astype(str).str.strip()

    # Clear columns values
    for col in ["attacking_type", "defending_type"]:
        # If a value is Nan break the run
        if type_chart[col].isna().any():
            nan_rows = type_chart[type_chart[col].isna()].index.tolist()
            raise ValueError(f"ERROR: found NaN value in column '{col}' at rows {nan_rows}.")

        # Remove white spaces and lower
        type_chart[col] = type_chart[col].astype(str).str.strip().str.lower()

    print(f"SUCCESS: cleaned {len(type_chart)} entries.")
    return type_chart

def load_type_chart(file_path: str) -> pd.DataFrame | None:
    """
    Read a type-chart CSV and return its normalized DataFrame
    """

    print(f"Loading data from '{file_path}'...")
    try:
        raw_df = pd.read_csv(file_path)
        return normalize_type_chart(raw_df)

    except FileNotFoundError:
        print(f"ERROR: could not find the file at '{file_path}'.")


# --- MULTPLIERS --- 
def get_multiplier(move_type: str, pkm_type1: str, pkm_type2: str) -> float:
    """
    Return attack effectiveness against both defensive types
    """

    m1 = type_chart.get((move_type, pkm_type1), 1.0) if pkm_type1 else 1.0
    m2 = type_chart.get((move_type, pkm_type2), 1.0) if pkm_type2 else 1.0
    return m1 * m2


# Getting type charts
type_chart = load_type_chart(TYPE_CHART_PATH)
df_indexed = type_chart.set_index(["attacking_type", "defending_type"])
type_chart = df_indexed["multiplier"].to_dict()
ATTACK_TYPES = {attack_type for attack_type, _ in type_chart}
stats_map = create_stats_map(train_data, test_data)

## 2. Extract Features (both Train and Test Features)

In [ ]:
# --- BASE STATS ---
def get_stats(turn: dict, p2_stats: dict) -> None:
    """
    Store player 2's active pokemon stats in p2_stats if found in stats_map
    """

    pkm2_name = turn["p2_pokemon_state"]["name"]
    if pkm2_name in stats_map:
        stats = stats_map[pkm2_name]["base_stats"]
        p2_stats[pkm2_name] = stats

def feature_base_stats(battle: dict, p2_stats: dict, features: dict) -> None:
    """
    Add base-stat for P1's full team and P2's known team.
    For each stat, compute sum, mean, min, max and population standard deviation
    """

    for player, team in (("p1", battle["p1_team_details"]), ("p2", p2_stats.values())):
        # Single values + merged values
        features[f"{player}_sum_bst"] = 0
        features[f"{player}_mean_bst"] = 0
        bst_values = np.zeros(len(team))  # One total per Pokemon with known stats
        for stat in STATISTICS:
            values = [pokemon[f"base_{stat}"] for pokemon in team]
            features[f"{player}_sum_{stat}"]  = np.sum(values)
            features[f"{player}_mean_{stat}"] = np.mean(values)
            features[f"{player}_min_base_{stat}"] = min(values)
            features[f"{player}_max_base_{stat}"] = max(values)
            features[f"{player}_std_base_{stat}"] = float(np.std(values, ddof=0))

            features[f"{player}_sum_bst"] += features[f"{player}_sum_{stat}"]
            bst_values += values

        # Merged values
        features[f"{player}_mean_bst"] = features[f"{player}_sum_bst"] / TEAM_SIZE
        features[f"{player}_std_base_bst"] = float(np.std(bst_values, ddof=0))

    # Differentials
    for stat in STATISTICS:
        features[f"diff_sum_{stat}"]  = features[f"p1_sum_{stat}"] - features[f"p2_sum_{stat}"]
        features[f"diff_mean_{stat}"] = features[f"p1_mean_{stat}"] - features[f"p2_mean_{stat}"]
    features["diff_sum_bst"]  = features["p1_sum_bst"]  - features["p2_sum_bst"]
    features["diff_mean_bst"] = features["p1_mean_bst"] - features["p2_mean_bst"]
    features["diff_std_base_bst"] = features["p1_std_base_bst"] - features["p2_std_base_bst"]


# --- TEAM TYPES ---
def feature_team_types(battle: dict, p2_stats: dict, features: dict) -> None:
    """
    Add known team type diversity and repeated defensive weaknesses
    """
    
    p1_team = [pokemon["name"] for pokemon in battle["p1_team_details"]]
    for player, team in (("p1", p1_team), ("p2", p2_stats)):
        team_types = []
        unique_types = set()
        for name in team:
            # If name is present then we collect the types related
            types = [type for type in stats_map.get(name, {}).get("types", [])]
            if types:
                # Ensure first 2 entries
                team_types.append(types)
                unique_types.update(types)

        # Number of pokemons with known types
        known_count = len(team_types)
        features[f"{player}_known_type_count"] = known_count
        features[f"{player}_type_diversity"] = len(unique_types)
        
        total_weakness_severity, shared_weakness_type_count = 0.0, 0
        for attack_type in ATTACK_TYPES:
            multipliers = [get_multiplier(attack_type, *types) for types in team_types]
            # Sum of (super-)effective moves off team based on attact_type
            total_weakness_severity += sum((multiplier // 2) if multiplier > 1.0 else 0.0 for multiplier in multipliers)
            # At least two pokemon are weak to the same type (multiplier > 1.0)
            shared_weakness_type_count += sum(multiplier > 1.0 for multiplier in multipliers) >= 2

        features[f"{player}_average_weakness_severity"] = total_weakness_severity / known_count if known_count else np.nan
        features[f"{player}_shared_weakness_type_count"] = shared_weakness_type_count if known_count else np.nan

    features["diff_type_diversity"] = features["p1_type_diversity"] - features["p2_type_diversity"]
    features["diff_average_weakness_severity"]  = features["p2_average_weakness_severity"] - features["p1_average_weakness_severity"]
    features["diff_shared_weakness_type_count"] = features["p2_shared_weakness_type_count"] - features["p1_shared_weakness_type_count"]


# --- HP LOSS ---
def get_hp_pct_lost(turn: dict, p1_hp_map: dict, p2_hp_map: dict) -> None:
    """
    Accumulate each observed Pokemon's net HP loss.
    Track its latest HP fraction across switches, assuming full HP before its first observation.
    Add previous HP minus current HP to the corresponding 1-10, 11-20 or 21-30 turn interval.
    Healing contributes negative loss
    """

    for player, hp_map in (("p1", p1_hp_map), ("p2", p2_hp_map)):
        name = turn[f"{player}_pokemon_state"]["name"]
        hp = turn[f"{player}_pokemon_state"]["hp_pct"]
        if name not in hp_map:
            hp_map[name] = {"last_hp": 1.0, "losses": [0.0, 0.0, 0.0]}
        pokemon = hp_map[name]
        pokemon["losses"][(turn["turn"] - 1) // 10] += pokemon["last_hp"] - hp
        pokemon["last_hp"] = hp

def feature_hp_lost(p1_hp_map: dict, p2_hp_map: dict, features: dict) -> None:
    """
    Add interval losses, total losses and per-Pokemon loss dispersion.
    Interval values and totals are sums of HP fractions.
    Compute population standard deviation from each Pokemon's total net loss, padding unseen team members with zero loss.
    Also compute differences P2 - P1 loss
    """

    for player, team in (("p1", p1_hp_map), ("p2", p2_hp_map)):
        # Single values + merged values
        features[f"{player}_sum_hp_lost"] = 0
        # Keep one total per Pokemon, with zero loss for unseen Pokemon.
        hp_lost_values = np.zeros(TEAM_SIZE)
        for i, interval in enumerate(HP_PCT_LOST_INTERVALS):
            values = [pokemon["losses"][i] for pokemon in team.values()]
            features[f"{player}_hp_pct_lost_{interval}"] = sum(values)
            features[f"{player}_sum_hp_lost"] += features[f"{player}_hp_pct_lost_{interval}"]
            hp_lost_values[:len(team)] += values

        # Merged values
        features[f"{player}_mean_hp_pct_lost"] = features[f"{player}_sum_hp_lost"] / TEAM_SIZE
        features[f"{player}_std_hp_pct_lost"] = float(np.std(hp_lost_values, ddof=0))

    # Differentials
    for interval in HP_PCT_LOST_INTERVALS:
        features[f"diff_hp_pct_lost_{interval}"] = features[f"p2_hp_pct_lost_{interval}"] - features[f"p1_hp_pct_lost_{interval}"]
    features["diff_sum_hp_pct_lost"]  = features["p2_sum_hp_lost"] - features["p1_sum_hp_lost"]
    features["diff_mean_hp_pct_lost"] = features["p2_mean_hp_pct_lost"] - features["p1_mean_hp_pct_lost"]
    features["diff_std_hp_pct_lost"]  = features["p2_std_hp_pct_lost"] - features["p1_std_hp_pct_lost"]


# --- TYPE EFFECTIVENESS ---
def get_mults(turn: dict, p1_mults: dict, p2_mults: dict) -> None:
    """
    Count the effectiveness of each move.
    Skip absent moves and status moves.
    Unknown P2 species use neutral defensive types
    """

    mv1 = turn["p1_move_details"]
    if mv1 is not None and mv1["category"] != "status":
        mv1_type = mv1["type"]
        pkm2_name = turn["p2_pokemon_state"]["name"]
        pkm2_stats = stats_map.get(pkm2_name)
        if pkm2_stats is not None:
            pkm2_types = pkm2_stats.get("types", ["notype", "notype"])
        else:
            pkm2_types = ["notype", "notype"]
        mult1 = get_multiplier(mv1_type, pkm2_types[0], pkm2_types[1])
        p1_mults[mult1] += 1

    mv2 = turn["p2_move_details"]
    if mv2 is not None and mv2["category"] != "status":
        mv2_type = mv2["type"]
        pkm1_name = turn["p1_pokemon_state"]["name"]
        pkm1_types = stats_map[pkm1_name]["types"]
        mult2 = get_multiplier(mv2_type, pkm1_types[0], pkm1_types[1])
        p2_mults[mult2] += 1

def feature_mults(p1_mults: dict, p2_mults: dict, features: dict) -> None:
    """
    Add multiplier bucket counts and summaries of recorded moves.

    Compute count, sum, min, max, mean and population standard deviation.
    Bucket differences use P2 - P1 below 1x and P1 - P2 at or above 1x.
    """

    for player, team in (("p1", p1_mults), ("p2", p2_mults)):
        # Single values + merged values
        values = []
        features[f"{player}_count_mult"] = 0
        features[f"{player}_sum_mult"] = 0
        for mult in MULTIPLIERS:
            value = team[mult]
            values.extend([mult] * value)
            features[f"{player}_{mult}_mult"] = value
            features[f"{player}_sum_mult"] += mult * value
            features[f"{player}_count_mult"] += value

        # Merged values
        features[f"{player}_min_mult"] = min(values) if values else np.nan
        features[f"{player}_max_mult"] = max(values) if values else np.nan
        features[f"{player}_mean_mult"] = features[f"{player}_sum_mult"] / features[f"{player}_count_mult"] if values else np.nan
        features[f"{player}_std_mult"] = float(np.std(values, ddof=0)) if values else np.nan

    # Differentials
    features["diff_count_mult"] = 0
    for mult in MULTIPLIERS:
        if mult < 1:
            features[f"diff_{mult}_mult"] = features[f"p2_{mult}_mult"] - features[f"p1_{mult}_mult"]
            features["diff_count_mult"] += features[f"diff_{mult}_mult"]
        else:
            features[f"diff_{mult}_mult"] = features[f"p1_{mult}_mult"] - features[f"p2_{mult}_mult"]
            features["diff_count_mult"] += features[f"diff_{mult}_mult"]
    features["diff_min_mult"] = features["p1_min_mult"] - features["p2_min_mult"]
    features["diff_max_mult"] = features["p1_max_mult"] - features["p2_max_mult"]
    features["diff_sum_mult"]  = features["p1_sum_mult"] - features["p2_sum_mult"]
    features["diff_mean_mult"] = features["p1_mean_mult"] - features["p2_mean_mult"]
    features["diff_std_mult"] = features["p1_std_mult"] - features["p2_std_mult"]


# --- STATUSES (NO FAINT) ---
def get_statuses(turn: dict, p1_status: dict, p2_status: dict) -> None:
    """
    Accumulate active-Pokemon status observations.
    Exclude nostatus and fnt.
    A status that persists across several observed
    turns contributes once on each turn
    """
    
    for player, team in (("p1", p1_status), ("p2", p2_status)):
        pkm_state = turn[f"{player}_pokemon_state"]["status"]
        if pkm_state != "nostatus" and pkm_state != "fnt":
            if pkm_state not in team:
                team[pkm_state] = 0
            team[pkm_state] += 1

def feature_statuses(p1_status_map: dict, p2_status_map: dict, n_turns: int, features: dict) -> None:
    """
    Add per-status counts, normalized fractions and status totals.
    Normalize each count and the total by the denominator n_turns.
    All status differences use P2 - P1
    """

    for player, team in (("p1", p1_status_map), ("p2", p2_status_map)):
        # Single values + merged values
        features[f"{player}_sum_statuses"] = 0
        for status in STATUSES:
            value = team.get(status, 0)
            features[f"{player}_{status}"] = value
            features[f"{player}_status_frac_{status}"] = value / n_turns
            features[f"{player}_sum_statuses"] += value

        # Merged values
        features[f"{player}_mean_statuses"] = features[f"{player}_sum_statuses"] / n_turns

    # Differentials
    for status in STATUSES:
        features[f"diff_{status}"] = features[f"p2_{status}"] - features[f"p1_{status}"]
        features[f"diff_status_frac_{status}"] = features[f"diff_{status}"] / n_turns
    features["diff_sum_statuses"]  = features["p2_sum_statuses"] - features["p1_sum_statuses"]
    features["diff_mean_statuses"] = features["p2_mean_statuses"] - features["p1_mean_statuses"]


# --- BOOSTS (CONSISTENT SCHEMA & DIFFERENTIALS) ---
def get_boosts(turn: dict, p1_boosts: dict, p2_boosts: dict) -> None:
    """
    Accumulate observed signed boost stages for both active Pokemon.
    Add each stat's current stage on every turn.
    Persistent boosts are counted repeatedly
    """

    for player, team in (("p1", p1_boosts), ("p2", p2_boosts)):
        for boost, value in turn[f"{player}_pokemon_state"]["boosts"].items():
            team[boost] = team.get(boost, 0) + value

def feature_boosts(p1_boosts: dict, p2_boosts: dict, n_turns: int, features: dict) -> None:
    """
    Add per-stat boost totals and signed aggregate summaries.
    Sum the supplied stage observations.
    Divide the total by the denominator n_turns.
    All boost differences use P1 - P2
    """

    for player, team in (("p1", p1_boosts), ("p2", p2_boosts)):
        # Single values + merged values
        features[f"{player}_sum_boosts"] = 0
        for boost in BOOSTS:
            value = team.get(boost, 0)
            features[f"{player}_{boost}_boost"] = value
            features[f"{player}_sum_boosts"] += value

        # Merged values
        features[f"{player}_mean_boosts"] = features[f"{player}_sum_boosts"] / n_turns

    # Differentials
    for boost in BOOSTS:
        features[f"diff_{boost}_boost"] = features[f"p1_{boost}_boost"] - features[f"p2_{boost}_boost"]
    features["diff_sum_boosts"]  = features["p1_sum_boosts"] - features["p2_sum_boosts"]
    features["diff_mean_boosts"] = features["p1_mean_boosts"] - features["p2_mean_boosts"]


# --- EFFECTS FEATURES ---
def get_effects(turn: dict, p1_effects: dict, p2_effects: dict) -> None:
    """
    Accumulate each active Pokemon's observed effect.
    Exclude noeffect.
    Persistent effects contribute repeatedly
    """

    for player, team in (("p1", p1_effects), ("p2", p2_effects)):
        for effect in turn[f"{player}_pokemon_state"]["effects"]:
            if effect != "noeffect":
                team[effect] = team.get(effect, 0) + 1

def feature_effects(p1_effects: dict, p2_effects: dict, n_turns: int, features: dict) -> None:
    """
    Add effect counts, good/bad totals and signed net summaries.
    Group effects as good or bad for the Pokemon carrying them.
    Net effects equal good counts minus bad counts.
    Normalize group and net totals by the denominator n_turns.
    Differences use P1 - P2 for good/net effects and P2 - P1 for bad effects
    """

    for player, team in (("p1", p1_effects), ("p2", p2_effects)):
        # Single values + merged values
        for group, effects in EFFECTS.items():
            features[f"{player}_sum_{group}_effects"] = 0
            for effect in effects:
                value = team.get(effect, 0)
                features[f"{player}_{effect}_effect"] = value
                features[f"{player}_sum_{group}_effects"] += value
            features[f"{player}_mean_{group}_effects"] = features[f"{player}_sum_{group}_effects"] / n_turns

        # Merged values: good effects minus bad effects
        features[f"{player}_sum_effects"]  = features[f"{player}_sum_good_effects"] - features[f"{player}_sum_bad_effects"]
        features[f"{player}_mean_effects"] = features[f"{player}_sum_effects"] / n_turns

    # Differentials for player 1 POV
    for group, effects in EFFECTS.items():
        first, second = ("p1", "p2") if group == "good" else ("p2", "p1")
        for effect in effects:
            features[f"diff_{effect}_effect"] = features[f"{first}_{effect}_effect"] - features[f"{second}_{effect}_effect"]
        features[f"diff_sum_{group}_effects"] = features[f"{first}_sum_{group}_effects"] - features[f"{second}_sum_{group}_effects"]
        features[f"diff_mean_{group}_effects"] = features[f"{first}_mean_{group}_effects"] - features[f"{second}_mean_{group}_effects"]
    features["diff_sum_effects"] = features["p1_sum_effects"] - features["p2_sum_effects"]
    features["diff_mean_effects"] = features["p1_mean_effects"] - features["p2_mean_effects"]


# --- TEAM SNAPSHOT AT LAST TURN ---
def get_last_seen(turn: dict, p1_last_seen: dict, p2_last_seen: dict) -> None:
    """
    Replace each observed Pokemon's latest HP/status.
    Keep the previous snapshots of switched-out Pokemon until they are observed again
    """

    for player, team in (("p1", p1_last_seen), ("p2", p2_last_seen)):
        pokemon = turn[f"{player}_pokemon_state"]
        team[pokemon["name"]] = {
            "hp_pct": pokemon["hp_pct"],
            "status": pokemon["status"],
        }

def feature_last_seen(last_turn: dict, p1_last_seen: dict, p2_last_seen: dict, features: dict) -> None:
    """
    Add latest team HP/status summaries and final active boosts/effects.
    Use each observed Pokemon's latest HP and status, which may be older for benched Pokemon.
    Assume unseen team members have full HP and no status.
    Exclude fnt from status counts.
    Boosts and effects come from the final active Pokemon only.
    Means divide by TEAM_SIZE.
    Pokemon slots follow first-observed order, followed by unseen team members.
    Differences use P1 - P2 for HP, boosts and good/net effects,
    and P2 - P1 for statuses and bad effects.
    """

    for player, team in (("p1", p1_last_seen), ("p2", p2_last_seen)):
        team = list(team.values())
        # Full health pokemon padding
        team.extend({"hp_pct": 1.0, "status": "nostatus"} for _ in range(TEAM_SIZE - len(team)))

        # Single values + merged values
        features[f"{player}_sum_remaining_hp_pct"] = 0
        features[f"{player}_sum_remaining_statuses"] = 0
        for status in STATUSES:
            features[f"{player}_remaining_{status}"] = 0
        hp_values = []
        for i, pokemon in enumerate(team):
            hp = pokemon["hp_pct"]
            hp_values.append(hp)
            features[f"{player}_{i}_pkm_remaining_hp_pct"] = hp
            features[f"{player}_sum_remaining_hp_pct"] += hp
            status = pokemon["status"]
            features[f"{player}_{i}_pkm_remaining_status"] = status
            if status in STATUSES:
                features[f"{player}_remaining_{status}"] += 1
                features[f"{player}_sum_remaining_statuses"] += 1

        # Merged values
        features[f"{player}_mean_remaining_hp_pct"] = features[f"{player}_sum_remaining_hp_pct"] / TEAM_SIZE
        features[f"{player}_std_remaining_hp_pct"] = float(np.std(hp_values, ddof=0))
        features[f"{player}_mean_remaining_statuses"] = features[f"{player}_sum_remaining_statuses"] / TEAM_SIZE

        active = last_turn[f"{player}_pokemon_state"]

        # Single values + merged values
        features[f"{player}_sum_remaining_boosts"] = 0
        for boost in BOOSTS:
            value = active["boosts"].get(boost, 0)
            features[f"{player}_remaining_{boost}_boost"] = value
            features[f"{player}_sum_remaining_boosts"] += value
        features[f"{player}_mean_remaining_boosts"] = features[f"{player}_sum_remaining_boosts"] / TEAM_SIZE

        # Single values + merged values
        for group, effects in EFFECTS.items():
            features[f"{player}_sum_remaining_{group}_effects"] = 0
            for effect in effects:
                # Whether the final active Pokemon currently has this effect.
                value = int(effect in active["effects"])
                features[f"{player}_remaining_{effect}_effect"] = value
                features[f"{player}_sum_remaining_{group}_effects"] += value
            features[f"{player}_mean_remaining_{group}_effects"] = features[f"{player}_sum_remaining_{group}_effects"] / TEAM_SIZE
        features[f"{player}_sum_remaining_effects"] = features[f"{player}_sum_remaining_good_effects"] - features[f"{player}_sum_remaining_bad_effects"]
        features[f"{player}_mean_remaining_effects"] = features[f"{player}_sum_remaining_effects"] / TEAM_SIZE

    # Differentials from player 1 POV
    features[f"diff_sum_remaining_hp_pct"]  = features[f"p1_sum_remaining_hp_pct"]  - features[f"p2_sum_remaining_hp_pct"]
    features[f"diff_mean_remaining_hp_pct"] = features[f"p1_mean_remaining_hp_pct"] - features[f"p2_mean_remaining_hp_pct"]
    features[f"diff_std_remaining_hp_pct"]  = features[f"p1_std_remaining_hp_pct"]  - features[f"p2_std_remaining_hp_pct"]

    for status in STATUSES:
        features[f"diff_remaining_{status}"] = features[f"p2_remaining_{status}"] - features[f"p1_remaining_{status}"]
    features[f"diff_sum_remaining_statuses"]  = features[f"p2_sum_remaining_statuses"]  - features[f"p1_sum_remaining_statuses"]
    features[f"diff_mean_remaining_statuses"] = features[f"p2_mean_remaining_statuses"] - features[f"p1_mean_remaining_statuses"]
    
    for boost in BOOSTS:
        features[f"diff_remaining_{boost}_boost"] = features[f"p1_remaining_{boost}_boost"] - features[f"p2_remaining_{boost}_boost"]
    features[f"diff_sum_remaining_boosts"]  = features[f"p1_sum_remaining_boosts"]  - features[f"p2_sum_remaining_boosts"]
    features[f"diff_mean_remaining_boosts"] = features[f"p1_mean_remaining_boosts"] - features[f"p2_mean_remaining_boosts"]

    for group, effects in EFFECTS.items():
        first, second = ("p1", "p2") if group == "good" else ("p2", "p1")
        for effect in effects:
            features[f"diff_remaining_{effect}_effect"] = features[f"{first}_remaining_{effect}_effect"] - features[f"{second}_remaining_{effect}_effect"]
        features[f"diff_sum_remaining_{group}_effects"]  = features[f"{first}_sum_remaining_{group}_effects"]  - features[f"{second}_sum_remaining_{group}_effects"]
        features[f"diff_mean_remaining_{group}_effects"] = features[f"{first}_mean_remaining_{group}_effects"] - features[f"{second}_mean_remaining_{group}_effects"]
        features[f"diff_sum_remaining_effects"]  = features[f"p1_sum_remaining_effects"]  - features[f"p2_sum_remaining_effects"]
        features[f"diff_mean_remaining_effects"] = features[f"p1_mean_remaining_effects"] - features[f"p2_mean_remaining_effects"]


# --- PLAYER 2 FAINT ---
def get_p2_faint_evidence(turn: dict, idx: int, timeline: list, tracker: dict) -> None:
    """
    Track confirmed P2 KOs and likely KOs at a Pokemon's final observed departure.
    Low-HP departures are evidence only; they do not confirm a KO
    """

    p1, p2 = (turn[f"{player}_pokemon_state"] for player in ("p1", "p2"))
    p1_move, p2_move = (turn.get(f"{player}_move_details") or {} for player in ("p1", "p2"))
    name, hp, status, effects = p2["name"], p2["hp_pct"], p2["status"], p2["effects"]

    if status == "fnt" or hp <= 0 or p2_move.get("name") in {"explosion", "selfdestruct"}:
        tracker["confirmed"].add(name)
        tracker["ghost"].discard(name)

    # The next active Pokemon identifies a departure in the recorded timeline.
    if (
        idx + 1 >= len(timeline)
        or name in tracker["confirmed"]
        or any(t["p2_pokemon_state"]["name"] == name for t in timeline[idx + 1:])
        or p1_move.get("name") in {"roar", "whirlwind"}
    ):
        return

    damaging = p1_move.get("base_power", 0) > 0 and p1_move.get("category") in {"physical", "special"}
    residual = hp <= 0.15 and (status in {"tox", "psn", "brn"} or "leechseed" in effects)
    destiny_bond = "destinybond" in effects or "destinybond" in p1["effects"]
    if residual or (destiny_bond and damaging) or (damaging and hp <= 0.40):
        tracker["ghost"].add(name)

def feature_p2_faint_evidence(tracker: dict, p1_last_seen: dict, p2_last_seen: dict, features: dict) -> None:
    """
    Add per-Pokemon KO flags and team sums, means and population standard deviations.
    P1 KOs come from observed faint/zero-HP states; P2 also includes inferred KOs.
    Slots follow first-observed order, with zero KO evidence for unseen Pokemon.
    KO differences use P2 - P1, equivalent to P1 - P2 for Pokemon still alive
    """

    for player, team in (("p1", p1_last_seen), ("p2", p2_last_seen)):
        confirmed = {name for name, pokemon in team.items() if pokemon["status"] == "fnt" or pokemon["hp_pct"] <= 0}
        if player == "p2":
            confirmed |= tracker["confirmed"]
        estimated = confirmed | tracker["ghost"] if player == "p2" else confirmed
        names = list(team) + [None] * (TEAM_SIZE - len(team))

        for metric, fainted in (("confirmed_kos", confirmed), ("estimated_true_kos", estimated)):
            values = [int(name in fainted) for name in names]
            for i, value in enumerate(values):
                features[f"{player}_{i}_pkm_{metric}"] = value
            for summary, value in (("sum", np.sum(values)), ("mean", np.mean(values)), ("std", np.std(values, ddof=0))):
                features[f"{player}_{summary}_{metric}"] = float(value)
            features[f"{player}_{metric}"] = features[f"{player}_sum_{metric}"]

        features[f"{player}_fnt"] = features[f"{player}_confirmed_kos"]
        features[f"{player}_estimated_pokemon_alive"] = TEAM_SIZE - features[f"{player}_estimated_true_kos"]
        features[f"{player}_known_remaining_hp_sum"] = sum(pokemon["hp_pct"] for pokemon in team.values()) + max(0, TEAM_SIZE - len(team))

    for metric in ("confirmed_kos", "estimated_true_kos"):
        for summary in ("sum", "mean", "std"):
            features[f"diff_{summary}_{metric}"] = features[f"p2_{summary}_{metric}"] - features[f"p1_{summary}_{metric}"]
        features[f"diff_{metric}"] = features[f"p2_{metric}"] - features[f"p1_{metric}"]
    features["diff_estimated_pokemon_alive"] = features["p1_estimated_pokemon_alive"] - features["p2_estimated_pokemon_alive"]


# --- HELD ITEMS ---
def get_items(turn: dict, p2_item_tracker: dict) -> None:
    """
    Collect P2 item evidence by Pokemon, counting repeated activations only once.
    A small HP gain on consecutive active turns suggests Leftovers, unless a healing
    move or Leech Seed can explain it. Explicit item effects are recorded directly
    """

    state = turn["p2_pokemon_state"]
    name, hp = state["name"], state["hp_pct"]
    move = turn.get("p2_move_details") or {}
    items = p2_item_tracker["seen"].setdefault(name, set())
    healing = move.get("name") in HEALING_MOVES
    if name == p2_item_tracker.get("prev_mon"):
        hp_gain = hp - p2_item_tracker["prev_hp"]
        if (
            0 < hp_gain <= 1 / 16 + 0.01
            and not (healing or p2_item_tracker.get("prev_healing", False))
            and "leechseed" not in turn["p1_pokemon_state"]["effects"]
        ):
            items.add("leftovers")

    for effect in state["effects"]:
        if effect in KNOWN_ITEMS:
            items.add(effect)
    p2_item_tracker.update(prev_mon=name, prev_hp=hp, prev_healing=healing)

def feature_items(battle: dict, p2_item_tracker: dict, features: dict) -> None:
    """
    Add per-Pokemon items and item-group indicators, with team sum/mean/std and diffs.
    P1 slots use team order; P2 slots use first-observed order and unknown padding.
    P2 zeros mean no detected item evidence, rather than confirmed absence of an item.
    Item-group differences use P1 - P2, matching the original item counts
    """

    p1_items = [{pokemon.get("item") or "noitem"} for pokemon in battle["p1_team_details"]]
    for player, team in (("p1", p1_items), ("p2", list(p2_item_tracker["seen"].values()))):
        team = team + [set() for _ in range(TEAM_SIZE - len(team))]
        for i, items in enumerate(team):
            features[f"{player}_{i}_pkm_item"] = "|".join(sorted(items)) if items else "unknown"

        for group in ITEM_METRICS:
            values = [
                int(bool(items) and "leftovers" not in items) if group == "non_leftovers"
                else int(bool(items & ITEM_GROUPS[group]))
                for items in team
            ]
            for i, value in enumerate(values):
                features[f"{player}_{i}_pkm_item_{group}"] = value
            for summary, value in (("sum", np.sum(values)), ("mean", np.mean(values)), ("std", np.std(values, ddof=0))):
                features[f"{player}_{summary}_item_{group}"] = float(value)
            features[f"{player}_item_{group}_count"] = features[f"{player}_sum_item_{group}"]

        features[f"{player}_item_has_all_leftovers"] = int(features[f"{player}_sum_item_leftovers"] == TEAM_SIZE)
        features[f"{player}_item_unique_count"] = len(set().union(*team))
        features[f"{player}_item_non_leftovers"] = features[f"{player}_sum_item_non_leftovers"]

    for group in ITEM_METRICS:
        for summary in ("sum", "mean", "std"):
            features[f"diff_{summary}_item_{group}"] = features[f"p1_{summary}_item_{group}"] - features[f"p2_{summary}_item_{group}"]
    for group in ("leftovers", "rng"):
        features[f"p2_detected_{group}_count"] = features[f"p2_sum_item_{group}"]
    for name, group in (("leftovers_count", "leftovers"), ("junk_items_count", "junk")):
        features[f"diff_{name}"] = features[f"diff_sum_item_{group}"]
    for name, species, item in (("thickclub_marowak", {"marowak", "cubone"}, "thickclub"), ("lightball_pikachu", {"pikachu"}, "lightball")):
        features[f"p1_has_{name}"] = int(any(pokemon["name"] in species and pokemon.get("item") == item for pokemon in battle["p1_team_details"]))


# --- DEPLOYED POKEMON, LEVELS & EFFECTIVE HP ---
def calc_eff_stat(base_val: float, level: int, is_hp: bool = False) -> float:
    """
    Estimate a Gen 2 stat from base stat and level, assuming max DV and StatExp
    """

    core = ((base_val + 47) * 2 * level) / 100.0
    return core + level + 10 if is_hp else core + 5

def get_deployed_turns(turn: dict, p1_deployed_turns: dict, p2_deployed_turns: dict) -> None:
    """
    Count active turns per observed Pokemon for both players.
    """

    for player, team in (("p1", p1_deployed_turns), ("p2", p2_deployed_turns)):
        name = turn[f"{player}_pokemon_state"]["name"]
        team[name] = team.get(name, 0) + 1

def feature_deployed_and_levels(battle: dict, p1_deployed_turns: dict, p2_deployed_turns: dict, p1_hp_map: dict, p2_hp_map: dict, features: dict) -> None:
    """
    Add per-Pokemon deployment, level and remaining effective HP, plus sum/mean/std.
    P1 slots follow team order; P2 slots follow first-observed order, then its known
    lead if unseen, then unknown Pokemon. P2 uses its known lead level and assumes
    level 100 otherwise. Unknown species use base HP 80; unseen slots use 85/full HP.
    Differences use P1 - P2. Deployment counts have no fixed good/bad meaning
    """

    p1_team = {pokemon["name"]: pokemon for pokemon in battle["p1_team_details"]}
    p2_lead = battle["p2_lead_details"]
    p2_names = list(dict.fromkeys([*p2_deployed_turns, p2_lead["name"]]))
    for player, names, deployed_turns, hp_map in (
        ("p1", list(p1_team), p1_deployed_turns, p1_hp_map),
        ("p2", p2_names, p2_deployed_turns, p2_hp_map),
    ):
        names = names + [None] * (TEAM_SIZE - len(names))
        values = {metric: [] for metric in DEPLOYED_METRICS}
        total_hp, weighted_level, deployed_levels = 0.0, 0.0, []
        for i, name in enumerate(names):
            if player == "p1":
                pokemon = p1_team[name]
            elif name == p2_lead["name"]:
                pokemon = p2_lead
            else:
                pokemon = {
                    "level": 100,
                    "base_hp": stats_map.get(name, {}).get("base_stats", {}).get("base_hp", 85 if name is None else 80),
                }
            level = pokemon["level"]
            turns = deployed_turns.get(name, 0)
            hp = calc_eff_stat(pokemon["base_hp"], level, is_hp=True)
            total_hp += hp
            weighted_level += level * turns
            if turns:
                deployed_levels.append(level)
            for metric, value in (
                ("deployed_turns", turns), ("level", level),
                ("remaining_effective_hp", hp * hp_map.get(name, {}).get("last_hp", 1.0)),
            ):
                values[metric].append(value)
                features[f"{player}_{i}_pkm_{metric}"] = value

        for metric, observations in values.items():
            for summary, value in (("sum", np.sum(observations)), ("mean", np.mean(observations)), ("std", np.std(observations, ddof=0))):
                features[f"{player}_{summary}_{metric}"] = float(value)
        features[f"{player}_deployed_count"] = len(deployed_turns)
        features[f"{player}_min_level"] = min(values["level"])
        features[f"{player}_underleveled_count"] = sum(level < 100 for level in values["level"])
        features[f"{player}_deployed_underleveled_count"] = sum(level < 100 for level in deployed_levels)
        features[f"{player}_deployed_min_lvl"] = min(deployed_levels, default=100)
        features[f"{player}_active_wmean_lvl"] = weighted_level / max(sum(deployed_turns.values()), 1)
        lead = p1_team.get(next(iter(deployed_turns), None), {}) if player == "p1" else p2_lead
        features[f"{player}_lead_level"] = lead.get("level", np.nan)
        features[f"{player}_rem_effective_hp_ratio"] = features[f"{player}_sum_remaining_effective_hp"] / total_hp if total_hp else 0.0

    for metric in DEPLOYED_METRICS:
        for summary in ("sum", "mean", "std"):
            features[f"diff_{summary}_{metric}"] = features[f"p1_{summary}_{metric}"] - features[f"p2_{summary}_{metric}"]
    for metric in DEPLOYED_SUMMARIES:
        features[f"diff_{metric}"] = features[f"p1_{metric}"] - features[f"p2_{metric}"]


# --- TACTICAL TEMPO, OFFENSIVE POWER, HAZARDS & REST ---
def get_tactics(turn: dict, p1_tactics: dict, p2_tactics: dict) -> None:
    """
    Record each turn's switches, moves, offensive power, RestTalk and phazing.
    Exclude replacements after observed KOs from voluntary switches.
    Offensive power is base power * STAB * type effectiveness * accuracy.
    Resolve Spikes/Rapid Spin for both players before recording hazard uptime
    """

    for player, opponent, tracker, opp_tracker in (
        ("p1", "p2", p1_tactics, p2_tactics),
        ("p2", "p1", p2_tactics, p1_tactics),
    ):
        state, opp_state = (turn[f"{side}_pokemon_state"] for side in (player, opponent))
        move = turn.get(f"{player}_move_details") or {}
        name = state["name"]
        fainted = state["status"] == "fnt" or state["hp_pct"] <= 0
        values = {metric: 0.0 for metric in TACTIC_METRICS}
        values["switches"] = int(tracker["prev_mon"] is not None and name != tracker["prev_mon"] and not tracker["prev_fainted"])
        tracker.update(prev_mon=name, prev_fainted=fainted)

        values["moves_used"] = int(bool(move))
        if move.get("category") in {"physical", "special"} and move.get("base_power", 0) > 0:
            attacker_types = stats_map.get(name, {}).get("types", ["notype", "notype"])
            defender_types = stats_map.get(opp_state["name"], {}).get("types", ["notype", "notype"])
            stab = STAB if move["type"] in attacker_types else 1.0
            values["eff_power"] = move["base_power"] * stab * move["accuracy"] * get_multiplier(move["type"], *defender_types)

        if not fainted:
            for metric, moves in TACTIC_MOVES.items():
                values[metric] = int(move.get("name") in moves)
            if move.get("name") == "spikes":
                tracker["spikes_active"] = 1
            elif move.get("name") == "rapidspin":
                opp_tracker["spikes_active"] = 0
        for metric, value in values.items():
            tracker["values"][metric].append(value)

    for tracker in (p1_tactics, p2_tactics):
        tracker["values"]["spikes_active"][-1] = tracker["spikes_active"]

def feature_tactics(p1_tactics: dict, p2_tactics: dict, n_turns: int, features: dict) -> None:
    """
    Add each tactical metric's sum, per-turn mean and population standard deviation.
    Include zero-action turns. Spikes sums count turns with that player's hazards up.
    Keep legacy names: moves/switches are rates, power is a mean, RestTalk/phazing
    are counts, and spikes_active is the final hazard flag. All diffs use P1 - P2
    """

    for player, tracker in (("p1", p1_tactics), ("p2", p2_tactics)):
        for metric in TACTIC_METRICS:
            values = tracker["values"][metric]
            total = sum(values)
            for summary, value in (("sum", total), ("mean", total / n_turns if n_turns else 0.0), ("std", np.std(values, ddof=0) if values else 0.0)):
                features[f"{player}_{summary}_{metric}"] = float(value)
            if metric in {"moves_used", "switches"}:
                features[f"{player}_{metric}"] = features[f"{player}_mean_{metric}"]
            elif metric == "eff_power":
                features[f"{player}_eff_power_mean"] = features[f"{player}_mean_eff_power"]
            elif metric == "spikes_active":
                features[f"{player}_{metric}"] = tracker["spikes_active"]
            else:
                features[f"{player}_{metric}"] = features[f"{player}_sum_{metric}"]

    for metric in TACTIC_METRICS:
        for summary in ("sum", "mean", "std"):
            features[f"diff_{summary}_{metric}"] = features[f"p1_{summary}_{metric}"] - features[f"p2_{summary}_{metric}"]
        legacy_metric = "eff_power_mean" if metric == "eff_power" else metric
        features[f"diff_{legacy_metric}"] = features[f"p1_{legacy_metric}"] - features[f"p2_{legacy_metric}"]


In [ ]:
# --- EXTRACT FEATURES ---
def extract_features(data: list) -> pd.DataFrame:
    """
    Extract comprehensive features combining absolute scale and differentials:
    1. base stats
    2. team weakness
    3. HP loss
    4. type multiplier
    5. statuses + p1 faint
    6. boosts
    7. effects
    8. last-turn snapshot
    10. p2 faint
    11. held items
    12. deployed Pokemon counts, levels and estimated remaining effective HP
    13. tactical tempo, effective offensive power, hazards, phazing & RestTalk
    """

    print("Extracting features...")
    feature_list = []
    for battle in data:
        features = {"battle_id": battle["battle_id"]}
        if "player_won" in battle:
            features["player_won"] = battle["player_won"]

        p1_hp_map, p2_hp_map = {}, {}
        p1_mults, p2_mults = ({mult: 0 for mult in MULTIPLIERS} for _ in range(2))
        p1_status, p2_status = {}, {}
        p1_boosts, p2_boosts = ({boost: 0 for boost in BOOSTS} for _ in range(2))
        p1_effects, p2_effects = {}, {}
        p1_last_seen, p2_last_seen = {}, {}
        p2_stats = {}
        p2_faint_tracker = {"confirmed": set(), "ghost": set()}
        p2_item_tracker = {"seen": {}, "prev_mon": None, "prev_hp": None, "prev_healing": False}
        p1_deployed_turns, p2_deployed_turns = {}, {}
        p1_tactics, p2_tactics = (
            {
                "prev_mon": None, "prev_fainted": False, "spikes_active": 0,
                "values": {metric: [] for metric in TACTIC_METRICS},
            }
            for _ in range(2)
        )

        timeline = battle["battle_timeline"]
        if not timeline:
            raise ValueError(f"Battle {battle['battle_id']} has no recorded turns.")
        for idx, turn in enumerate(timeline):
            get_stats(turn, p2_stats)
            get_hp_pct_lost(turn, p1_hp_map, p2_hp_map)
            get_mults(turn, p1_mults, p2_mults)
            get_statuses(turn, p1_status, p2_status)
            get_boosts(turn, p1_boosts, p2_boosts)
            get_effects(turn, p1_effects, p2_effects)
            get_last_seen(turn, p1_last_seen, p2_last_seen)
            get_p2_faint_evidence(turn, idx, timeline, p2_faint_tracker)
            get_items(turn, p2_item_tracker)
            get_deployed_turns(turn, p1_deployed_turns, p2_deployed_turns)
            get_tactics(turn, p1_tactics, p2_tactics)

        n_turns = len(timeline)
        # 1. base stats
        feature_base_stats(battle, p2_stats, features)
        # 2. team weakness
        feature_team_types(battle, p2_stats, features)
        # 3. HP loss
        feature_hp_lost(p1_hp_map, p2_hp_map, features)
        # 4. type multiplier
        feature_mults(p1_mults, p2_mults, features)
        # 5. statuses + p1 faint
        feature_statuses(p1_status, p2_status, n_turns, features)
        # 6. boosts
        feature_boosts(p1_boosts, p2_boosts, n_turns, features)
        # 7. effects
        feature_effects(p1_effects, p2_effects, n_turns, features)
        # 8. last-turn snapshot
        feature_last_seen(timeline[-1], p1_last_seen, p2_last_seen, features)
        # 10. p2 faint
        feature_p2_faint_evidence(p2_faint_tracker, p1_last_seen, p2_last_seen, features)
        # 11. held items
        feature_items(battle, p2_item_tracker, features)
        # 12. deployed Pokemon counts, levels and estimated remaining effective HP
        feature_deployed_and_levels(battle, p1_deployed_turns, p2_deployed_turns, p1_hp_map, p2_hp_map, features)
        # 13. tactical tempo, effective offensive power, hazards, phazing & RestTalk
        feature_tactics(p1_tactics, p2_tactics, n_turns, features)
        
        feature_list.append(features)

    print(f"SUCCESS: extracted {len(feature_list[0])} features.")
    return pd.DataFrame(feature_list).fillna(0)


# Extracting training and test features
train_df = extract_features(train_data)
print(f"Training features: {train_df.shape}.")
display(train_df.head())

test_df = extract_features(test_data)
print(f"Test features: {test_df.shape}.")
# display(test_df.head())

## 3. Training

### Models

In [ ]:
def build_model( random_state: int = RANDOM_STATE, iterations: int | None = None ) -> CatBoostClassifier:
    """
    Create a fresh CatBoost classifier with the configured settings
    """

    params = {**CATBOOST_PARAMS, "random_seed": random_state}
    if iterations is not None:
        params["iterations"] = iterations
    return CatBoostClassifier(**params)


def prepare_features(train_df: pd.DataFrame, test_df: pd.DataFrame) -> tuple:
    """
    Align features and normalize categorical values without using labels
    """

    if USE_FEATURE_SELECTION:
        feature_cols = list(dict.fromkeys(ALL_FEATURES))
    elif USE_ALLOWED_FEATURES:
        feature_cols = list(dict.fromkeys(ALLOWED_FEATURES))
    else:
        feature_cols = [
            col for col in train_df.columns
            if col not in {"battle_id", "player_won"}
        ]
    if not feature_cols:
        raise ValueError("No predictors selected. Check extraction and the configured feature lists.")
    if {"battle_id", "player_won"}.intersection(feature_cols):
        raise ValueError("Identifiers and target labels cannot be predictors.")
    for name, frame in (("training", train_df), ("test", test_df)):
        missing = sorted(set(feature_cols) - set(frame.columns))
        if missing:
            message = f"Missing {name} features: {missing}"
            if USE_FEATURE_SELECTION:
                message += (
                    ". Feature selection requires every entry in ALL_FEATURES. "
                    "Enable the relevant feature extraction calls and rerun extraction, "
                    "or explicitly restrict ALL_FEATURES to the intended candidate pool."
                )
            raise ValueError(message)
    unknown_categorical = sorted(set(CATEGORICAL_FEATURES) - set(feature_cols))
    if unknown_categorical:
        raise ValueError(f"Categorical columns must be selected: {unknown_categorical}")

    X = train_df.loc[:, feature_cols].copy()
    X_test = test_df.loc[:, feature_cols].copy()
    y = train_df["player_won"].copy()
    if y.isna().any() or not y.isin([0, 1]).all():
        raise ValueError("player_won must contain only 0 and 1.")
    y = y.astype(int)

    categorical_features = [
        col for col in feature_cols
        if col in CATEGORICAL_FEATURES
        or not pd.api.types.is_numeric_dtype(X[col])
    ]
    for frame in (X, X_test):
        for col in feature_cols:
            if col in categorical_features:
                frame[col] = frame[col].astype("string").fillna("__unknown__").astype(str)
            else:
                # CatBoost handles missing numeric values natively.
                frame[col] = pd.to_numeric(frame[col], errors="raise").replace(
                    [np.inf, -np.inf], np.nan
                )

    print(f"Selected features: {len(feature_cols)}; categorical: {len(categorical_features)}")
    return X, y, X_test, categorical_features

### Performance metrics

In [ ]:
# --- METRICS ---
def compute_metrics(y_true: pd.Series | np.ndarray, y_pred: np.ndarray, y_prob: np.ndarray) -> dict:
    """
    Return accuracy, precision, recall, F1, and ROC-AUC for binary predictions.
    y_true and y_pred contain 0/1 labels.
    y_prob contains probabilities of class 1
    """

    return {
        # (TP + TN) / total
        "accuracy": accuracy_score(y_true, y_pred),
        # TP / (TP + FP)
        "precision": precision_score(y_true, y_pred, zero_division=0),
        # TP / (TP + FN)
        "recall": recall_score(y_true, y_pred, zero_division=0),
        # 2 * (precision * recall) / (precision + recall) = 2 * TP / (2 * TP + FP + FN)
        "f1": f1_score(y_true, y_pred, zero_division=0),
        # Area Under the Receiver Operating Characteristic Curve
        "roc_auc": roc_auc_score(y_true, y_prob)
    }

### Training

In [ ]:
def select_feature_subset(X, y, categorical_features, random_state=RANDOM_STATE, scope="final"):
    """Batch backward elimination; all decisions use only the supplied training data."""
    if not 0 < SELECTION_DROP_FRACTION < 1:
        raise ValueError("SELECTION_DROP_FRACTION must be between 0 and 1.")
    if not 1 <= SELECTION_MIN_FEATURES <= X.shape[1]:
        raise ValueError("SELECTION_MIN_FEATURES must be between 1 and the candidate count.")
    if SELECTION_MAX_ITERATIONS < 1 or SELECTION_SHAP_SAMPLE_SIZE < 1:
        raise ValueError("Selection iterations and SHAP sample size must be positive.")
    if SELECTION_ACCURACY_TOLERANCE < 0:
        raise ValueError("SELECTION_ACCURACY_TOLERANCE cannot be negative.")

    # Reuse these three disjoint partitions at every elimination step.
    inner_train, tune_idx = train_test_split(
        np.arange(len(X)), test_size=SELECTION_VALIDATION_FRACTION,
        stratify=y, random_state=random_state,
    )
    fit_idx, stop_idx = train_test_split(
        inner_train, test_size=EARLY_STOPPING_FRACTION,
        stratify=y.iloc[inner_train], random_state=random_state,
    )
    shap_idx = np.random.default_rng(random_state).choice(
        fit_idx, size=min(SELECTION_SHAP_SAMPLE_SIZE, len(fit_idx)), replace=False,
    )
    active = X.columns.tolist()
    variable = X.iloc[fit_idx].nunique(dropna=False).gt(1)
    if not variable.any():
        raise ValueError("All candidate features are constant in the inner fitting data.")
    records = []
    selection_sizes = [len(active)]
    while selection_sizes[-1] > SELECTION_MIN_FEATURES:
        selection_sizes.append(max(
            SELECTION_MIN_FEATURES,
            int(selection_sizes[-1] * (1 - SELECTION_DROP_FRACTION)),
        ))
    selection_progress = tqdm(
        selection_sizes, desc=f"Feature selection: {scope}", unit="subset", leave=False,
    )
    for _ in selection_progress:
        cats = [col for col in categorical_features if col in active]
        model = build_model(
            random_state=random_state,
            iterations=min(SELECTION_MAX_ITERATIONS, CATBOOST_PARAMS["iterations"]),
        )
        model.fit(
            X.iloc[fit_idx].loc[:, active], y.iloc[fit_idx], cat_features=cats,
            eval_set=(X.iloc[stop_idx].loc[:, active], y.iloc[stop_idx]),
            early_stopping_rounds=EARLY_STOPPING_ROUNDS, use_best_model=True,
        )
        probability = model.predict_proba(X.iloc[tune_idx].loc[:, active])[:, 1]
        accuracy = accuracy_score(y.iloc[tune_idx], probability >= DECISION_THRESHOLD)
        records.append({
            "scope": scope,
            "feature_count": len(active),
            "inner_tuning_accuracy": accuracy,
            "iterations": model.tree_count_,
            "features": json.dumps(active),
        })
        selection_progress.set_postfix(features=len(active), inner_acc=f"{accuracy:.4f}")
        if len(active) == SELECTION_MIN_FEATURES:
            continue

        # Recompute SHAP after each batch removal so redundant features can take over.
        shap_values = model.get_feature_importance(
            Pool(X.iloc[shap_idx].loc[:, active], cat_features=cats),
            type="ShapValues", shap_calc_type="Approximate",
        )
        importance = np.abs(shap_values[:, :-1]).mean(axis=0)
        # Zero SHAP ties must not leave a constant-only subset.
        importance[~variable.loc[active].to_numpy()] = -np.inf
        keep_count = max(
            SELECTION_MIN_FEATURES,
            int(len(active) * (1 - SELECTION_DROP_FRACTION)),
        )
        keep = {active[i] for i in np.argsort(-importance, kind="stable")[:keep_count]}
        active = [col for col in active if col in keep]

    history = pd.DataFrame(records)
    best_score = history["inner_tuning_accuracy"].max()
    eligible = history[
        history["inner_tuning_accuracy"] >= best_score - SELECTION_ACCURACY_TOLERANCE
    ]
    chosen = eligible["feature_count"].idxmin()
    history["selected"] = history.index == chosen
    selected = json.loads(history.loc[chosen, "features"])
    tqdm.write(f"  {scope}: selected {len(selected)} features (best inner score: {best_score:.4f}).")
    return selected, history


def train_and_evaluate_cv(
    X: pd.DataFrame, y: pd.Series, categorical_features: list,
    k: int = K_FOLDS, n_repeats: int = N_REPEATS,
    random_state: int = RANDOM_STATE, use_feature_selection: bool = False,
) -> tuple:
    """
    Evaluate the complete selection/training procedure on untouched outer folds
    """

    splitter = RepeatedStratifiedKFold(
        n_splits=k, n_repeats=n_repeats, random_state=random_state
    )
    oof_probs = np.zeros(len(X))
    oof_counts = np.zeros(len(X), dtype=int)
    importances = pd.Series(0.0, index=X.columns)
    records = []
    selection_histories = []
    print(f"Starting {n_repeats} x {k}-fold CatBoost validation.")

    fold_progress = tqdm(
        splitter.split(X, y), total=k * n_repeats, desc="CatBoost CV", unit="fold",
    )
    for i, (train_idx, val_idx) in enumerate(fold_progress):
        repeat, fold = divmod(i, k)
        seed = random_state + i
        selected = X.columns.tolist()
        if use_feature_selection:
            selected, history = select_feature_subset(
                X.iloc[train_idx], y.iloc[train_idx], categorical_features,
                random_state=seed, scope=f"repeat_{repeat + 1}_fold_{fold + 1}",
            )
            selection_histories.append(history)
        fold_X = X.loc[:, selected]
        cats = [col for col in categorical_features if col in selected]
        fit_idx, stop_idx = train_test_split(
            train_idx,
            test_size=EARLY_STOPPING_FRACTION,
            stratify=y.iloc[train_idx],
            random_state=seed,
        )

        # The scored validation fold never participates in early stopping.
        stopping_model = build_model(random_state=seed)
        stopping_model.fit(
            fold_X.iloc[fit_idx], y.iloc[fit_idx],
            cat_features=cats,
            eval_set=(fold_X.iloc[stop_idx], y.iloc[stop_idx]),
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            use_best_model=True,
        )
        iterations = max(1, stopping_model.get_best_iteration() + 1)
        model = build_model(random_state=seed, iterations=iterations)
        model.fit(fold_X.iloc[train_idx], y.iloc[train_idx], cat_features=cats)

        train_prob = model.predict_proba(fold_X.iloc[train_idx])[:, 1]
        val_prob = model.predict_proba(fold_X.iloc[val_idx])[:, 1]
        train_metrics = compute_metrics(
            y.iloc[train_idx], (train_prob >= DECISION_THRESHOLD).astype(int), train_prob
        )
        val_metrics = compute_metrics(
            y.iloc[val_idx], (val_prob >= DECISION_THRESHOLD).astype(int), val_prob
        )
        oof_probs[val_idx] += val_prob
        oof_counts[val_idx] += 1
        importances.loc[selected] += model.feature_importances_
        records.append({
            "repeat": repeat + 1,
            "fold": fold + 1,
            "iterations": iterations,
            "feature_count": len(selected),
            "selected_features": json.dumps(selected),
            **{f"train_{name}": value for name, value in train_metrics.items()},
            **{f"val_{name}": value for name, value in val_metrics.items()},
        })
        fold_progress.set_postfix(features=len(selected), val_acc=f"{val_metrics['accuracy']:.4f}")
        tqdm.write(
            f"Repeat {repeat + 1}/{n_repeats} | Fold {fold + 1}/{k} | "
            f"Features: {len(selected)} | Trees: {iterations} | Train accuracy: {train_metrics['accuracy']:.4f} | "
            f"Validation accuracy: {val_metrics['accuracy']:.4f} | "
            f"ROC-AUC: {val_metrics['roc_auc']:.4f}"
        )

    if not np.all(oof_counts == n_repeats):
        raise RuntimeError("Every battle must receive one OOF prediction per repeat.")
    oof_probs /= oof_counts
    cv_results = pd.DataFrame(records)
    oof_metrics = compute_metrics(
        y, (oof_probs >= DECISION_THRESHOLD).astype(int), oof_probs
    )
    summary = pd.DataFrame({
        "train_mean": {name: cv_results[f"train_{name}"].mean() for name in oof_metrics},
        "validation_mean": {name: cv_results[f"val_{name}"].mean() for name in oof_metrics},
        "validation_std": {name: cv_results[f"val_{name}"].std(ddof=0) for name in oof_metrics},
        "oof": oof_metrics,
    })
    print("\nCatBoost validation summary (accuracy is the competition metric):")
    display(summary)
    feature_importance = pd.Series(
        importances / len(records), index=X.columns, name="mean_cv_importance"
    ).sort_values(ascending=False)
    selection_history = (
        pd.concat(selection_histories, ignore_index=True) if selection_histories else pd.DataFrame()
    )
    return cv_results, oof_probs, feature_importance, summary, selection_history


def export_run_report(
    summary: pd.DataFrame,
    used_features: list[str],
    use_allowed_features: bool,
    output_dir: str | Path = REPORT_DIR,
    selection_history: pd.DataFrame | None = None,
    cv_results: pd.DataFrame | None = None,
    candidate_features: list[str] | None = None,
) -> Path:
    """Save the displayed validation summary and the actual selected predictors."""
    created_at = datetime.now(timezone.utc)
    report_dir = Path(output_dir)
    report_dir.mkdir(parents=True, exist_ok=True)
    report_path = report_dir / f"catboost_report_{created_at:%Y%m%dT%H%M%S_%fZ}.md"

    lines = [
        "# CatBoost run report",
        "",
        f"Created (UTC): {created_at:%Y-%m-%d %H:%M:%S}",
        "",
        "## 1. CatBoost validation summary",
        "",
        "Accuracy is the competition metric. Validation std uses ddof=0.",
        "",
        "| Metric | " + " | ".join(summary.columns) + " |",
        "| --- | " + " | ".join("---:" for _ in summary.columns) + " |",
    ]
    for metric, row in summary.iterrows():
        lines.append(
            f"| {metric} | " + " | ".join(f"{value:.6f}" for value in row) + " |"
        )
    lines.extend([
        "",
        "## 2. Used features",
        "",
        f"ALLOWED_FEATURES filter: {'enabled' if use_allowed_features else 'disabled'}",
        f"Feature count: {len(used_features)}",
        "",
    ])
    lines.extend(f"- `{feature}`" for feature in used_features)
    if selection_history is not None and not selection_history.empty:
        prefix = report_path.with_suffix("")
        history_path = prefix.with_name(prefix.name + "_selection_history.csv")
        folds_path = prefix.with_name(prefix.name + "_folds.csv")
        stability_path = prefix.with_name(prefix.name + "_feature_stability.csv")
        selection_history.to_csv(history_path, index=False)
        cv_results.to_csv(folds_path, index=False)
        fold_features = cv_results["selected_features"].map(json.loads)
        stability = pd.DataFrame({
            "feature": candidate_features,
            "selected_folds": [
                sum(feature in subset for subset in fold_features)
                for feature in candidate_features
            ],
        })
        stability["selection_frequency"] = stability["selected_folds"] / len(cv_results)
        stability["in_final_subset"] = stability["feature"].isin(used_features)
        stability.sort_values(
            ["selection_frequency", "feature"], ascending=[False, True]
        ).to_csv(stability_path, index=False)
        unavailable = [col for col in dict.fromkeys(ALL_FEATURES) if col not in candidate_features]
        final_choice = selection_history[
            (selection_history["scope"] == "final") & selection_history["selected"]
        ].iloc[0]
        lines.extend([
            "", "## 3. Feature selection", "",
            "Method: SHAP-guided batch backward elimination, nested inside outer CV.",
            "Validation mean evaluates the selection procedure. Each outer fold may use a different subset.",
            "The final subset above is selected using all training labels; its inner score is a tuning score, not independent validation.",
            "This approximate path does not test every combination or guarantee the globally best subset.",
            "",
            f"Candidate features actually searched: {len(candidate_features)}",
            f"ALL_FEATURES not extracted: {len(unavailable)}",
            f"Fraction removed per step: {SELECTION_DROP_FRACTION}",
            f"Minimum features: {SELECTION_MIN_FEATURES}",
            f"Search tree limit: {SELECTION_MAX_ITERATIONS}",
            f"SHAP sample limit: {SELECTION_SHAP_SAMPLE_SIZE} (Approximate SHAP)",
            f"Inner tuning fraction: {SELECTION_VALIDATION_FRACTION}",
            f"Accuracy tolerance favoring smaller subsets: {SELECTION_ACCURACY_TOLERANCE}",
            f"Final subset inner tuning accuracy (optimistic): {final_choice['inner_tuning_accuracy']:.6f}",
            "",
            f"[All search steps]({history_path.name})",
            f"[Outer fold metrics and subsets]({folds_path.name})",
            f"[Selection frequency by feature]({stability_path.name})",
        ])
        if unavailable:
            lines.extend(["", "Unavailable ALL_FEATURES (extraction unchanged):", ""])
            lines.extend(f"- `{feature}`" for feature in unavailable)
    report_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    print(f"Run report saved to '{report_path.resolve()}'.")
    return report_path


X_train, y_train, X_test, categorical_features = prepare_features(train_df, test_df)
candidate_features = X_train.columns.tolist()
cv_results, oof_probabilities, feature_importance, validation_summary, selection_history = train_and_evaluate_cv(
    X_train, y_train, categorical_features, use_feature_selection=USE_FEATURE_SELECTION,
)

selected_features = candidate_features
if USE_FEATURE_SELECTION:
    # The final subset is chosen without using the outer validation scores.
    selected_features, final_history = select_feature_subset(
        X_train, y_train, categorical_features, scope="final",
    )
    selection_history = pd.concat([selection_history, final_history], ignore_index=True)
    X_train = X_train.loc[:, selected_features]
    X_test = X_test.loc[:, selected_features]
    categorical_features = [col for col in categorical_features if col in selected_features]
    print(f"\nFinal selected features ({len(selected_features)}): {selected_features}")

final_iterations = max(1, int(np.median(cv_results["iterations"])))
print(f"\nFitting one final CatBoost model on {len(X_train)} battles with {final_iterations} trees.")
final_model = build_model(iterations=final_iterations)
final_model.fit(X_train, y_train, cat_features=categorical_features)
test_probabilities = final_model.predict_proba(X_test)[:, 1]
test_predictions = (test_probabilities >= DECISION_THRESHOLD).astype(int)

report_path = export_run_report(
    validation_summary, selected_features,
    use_allowed_features=USE_ALLOWED_FEATURES and not USE_FEATURE_SELECTION,
    selection_history=selection_history, cv_results=cv_results,
    candidate_features=candidate_features,
)

## 4. Submission File

In [ ]:
def predict_and_save(predictions: np.ndarray, test_battle_ids: pd.Series, output_path: str = SUBMISSION_PATH, sample_path: str = SAMPLE_SUBMISSION_PATH) -> None:
    """
    Format and save the submission file using the required template
    """
    
    print(f"Saving predictions to '{output_path}'...")
    sample_submission = pd.read_csv(sample_path)
    expected_columns = ["battle_id", "player_won"]
    if list(sample_submission.columns) != expected_columns:
        raise ValueError(f"ERROR: expected submission columns {expected_columns}, got {list(sample_submission.columns)}")

    predictions_by_id = pd.Series(predictions, index=test_battle_ids)
    sample_submission["player_won"] = sample_submission["battle_id"].map(predictions_by_id)

    if sample_submission["player_won"].isna().any():
        raise ValueError(f"ERROR: some battle IDs in the '{sample_path}' have no matching prediction.")

    sample_submission["player_won"] = sample_submission["player_won"].astype(int)
    sample_submission.to_csv(output_path, index=False)

    print(f"SUCCESS: file '{output_path}' created.")
    print(sample_submission.head(10))
    print(f"Prediction distribution (0 vs 1): {dict(sample_submission['player_won'].value_counts())}")


# Submission file
predict_and_save(test_predictions, test_df["battle_id"], SUBMISSION_PATH, SAMPLE_SUBMISSION_PATH)

## Debug

In [ ]:
display(feature_importance.head(25).to_frame())

if DEBUG:
    display(cv_results)
    display(X_train.describe(include="all"))